# Eight-window features for within-subject inner-speech classification

Three independently optimized procedures: **classical ML**, **scratch EEG deep learning**,
and **pretrained LaBraM**. This notebook contains all implementation code and requires no
earlier notebook or companion Python file. It does not reproduce a paper's pipeline.

The task is four-class classification of **Up, Down, Right, Left** using the inner-speech
condition of OpenNeuro ds003626 / the Thinking Out Loud processed derivatives.
The objective is the highest reliable subject-specific performance found within the stated
search budget. **No accuracy threshold, including 50%, is promised.**

## Start here

1. Attach the processed EEG dataset to Kaggle and enable a T4 GPU for CUDA SVM.
   Skip CUDA installation if cuML/CuPy already work. LaBraM needs Internet once
   to download its checkpoint if you enable that pipeline.
2. If packages are missing, enable and run the installation cell, then restart the kernel.
3. In **Easy settings**, verify `DATA_ROOT`, keep `PROFILE="smoke"`, and set
   `RUN_EXPERIMENTS=True`. Classical ML is enabled by default; both neural procedures remain available.
4. Run all cells. Smoke mode checks one subject, 128 channels, two outer/two inner folds,
   and three neural epochs if you enable the neural procedures. Its score is not a performance estimate.
5. Use `PROFILE="pilot"` for a real five-fold run on subject 1. Use `PROFILE="full"`
   for all subjects, broader searches, and three neural refit seeds. Full evaluation is expensive;
   the preflight table shows the number of fits before training.
6. Download the **entire output directory**, including `checkpoints`, `frozen_features`,
   `pretrained`, `splits`, and `runs`. CSV results alone cannot restore models.

`(128,)` means **only 128 channels**. Use `(128, 64, 32, 16)` to run all four counts.
The comma in a one-element tuple is required.

This version adds eight-window AR + wavelet entropy + wavelet variance, bandpower fusion, and CUDA SVM fitting. Previous notebooks are unchanged. Start with a CUDA smoke run, then pilot.

## Dependencies

Keep Kaggle's working PyTorch/CUDA installation. The optional installation cell supplies
the EEG libraries and checkpoint integration. Restart after an install if an import or ABI
error appears; do not mix an already-imported NumPy with a newly installed binary package.

Feature extraction and LDA run on CPU; SVM fitting defaults to CUDA. LaBraM feature extraction and both neural architectures use
CUDA automatically when available. CUDA SVM uses RAPIDS/cuML, with an explicit CPU mode available. No remote training job or paid service is launched.

PyWavelets is required for the new extractor. Burg AR estimation is vectorized NumPy code; statsmodels is not a runtime dependency.

In [ ]:
INSTALL_EXTRA_PACKAGES = False
if INSTALL_EXTRA_PACKAGES:
    import subprocess, sys
    subprocess.check_call([
        sys.executable, "-m", "pip", "install",
        "mne>=1.6,<2", "pyriemann>=0.7,<0.13", "braindecode[hub]==1.8.1",
        "scikit-learn>=1.4,<1.11", "cloudpickle>=3,<4", "tqdm", "matplotlib", "PyWavelets>=1.6,<2"
    ])
    print("Installation complete. Restart the kernel, set INSTALL_EXTRA_PACKAGES=False, then Run All.")

## Optional CUDA SVM installation

Enable a Kaggle T4. If cuML/CuPy already work, skip installation. Otherwise enable this
switch, install matching CUDA wheels with Internet enabled, restart the kernel, and turn
the switch off. Preserve Kaggle's working PyTorch installation.
[RAPIDS installation guide](https://docs.nvidia.com/datascience/install/).


In [ ]:
INSTALL_CUDA_SVM = False  # only when cuML/CuPy is missing or incompatible
CUDA_MAJOR = 12           # match the runtime/toolkit; not the GPU model
if INSTALL_CUDA_SVM:
    import subprocess, sys, platform
    if platform.system() != "Linux":
        raise RuntimeError("Install/run RAPIDS on Linux Kaggle GPU, not macOS.")
    if CUDA_MAJOR not in (12, 13):
        raise ValueError("Choose CUDA_MAJOR 12 or 13 to match the runtime.")
    subprocess.check_call([
        sys.executable, "-m", "pip", "install",
        f"cuml-cu{CUDA_MAJOR}>=25.4,<27", f"cupy-cuda{CUDA_MAJOR}x",
        "--extra-index-url", "https://pypi.nvidia.com",
    ])
    print("Restart the kernel after installation. Leave this switch False, then Run All.")

## Easy settings - edit this cell

Classical ML is enabled by default to compare the existing features against the new eight-window
features. Set both neural switches to True if you also want to run the scratch and pretrained
procedures. Pipelines run sequentially. This version writes to its own output directory.

`CV_SEEDS` repeat trial splits; neural ensemble seeds repeat final model initializations.
`CLASSICAL_FEATURE_SET` controls baseline versus new feature candidates.


In [ ]:
PROFILE = "smoke"  # smoke -> pilot -> full
RUN_EXPERIMENTS = False  # change to True when ready

RUN_CLASSICAL = True
RUN_EEGNET = False       # scratch procedure: temporal EEGNet versus compact filter-bank CNN
RUN_LABRAM = False       # smoke: frozen probe; pilot/full also test partial/full fine-tuning

DATA_ROOT = "/kaggle/input/datasets/truthisneverlinear/inner-speech-recognition/inner-speech-recognition/derivatives"
OUTPUT_ROOT = "/kaggle/working/inner_speech_eight_window_cuda"

CHANNEL_COUNTS = (128,)
SUBJECT_IDS = None if PROFILE == "full" else (1,)  # None = all available subjects
CV_SEEDS = (42,)
RESUME = True

CLASSICAL_FEATURE_SET = "combined"  # combined, eight_window, or baseline
SVM_BACKEND = "cuda"  # mandatory cuML/CuPy GPU fitting; explicit cpu is available

## What is being compared

| Procedure | Candidate representation/model | Fitted only on training trials |
|---|---|---|
| Classical | Eight-window AR/entropy/variance, optionally fused with bandpower; time-binned log bandpower; spatial-PCA + multiclass CSP; spatial-PCA + OAS covariance + Riemannian tangent space | Spatial projections, CSP, tangent reference, feature reduction, scaling, SVM/LDA |
| Scratch DL | EEGNet-inspired convolutional network with ordered temporal bins; compact FBCNet-inspired challenger | Channel scaling, weights, configuration, early-stopping duration |
| Pretrained | Frozen LaBraM features + PCA/logistic regression; last-two-block adaptation; full adaptation | Probe scaling/PCA/classifier, or adapted weights; winning strategy selected internally |

Classical ML selects between linear/RBF SVM and shrinkage LDA where enabled by the profile.
Scratch DL is still **one experimental procedure**, even if a different candidate wins for
different subjects/folds. The chosen model is recorded; do not label every scratch result
as standard EEGNet. The two scratch architectures are explicitly defined adaptations.

Only target-subject trials train the classical and scratch models. LaBraM starts from an
external pretrained checkpoint. This version does not add training from the other nine
dataset subjects, so that source of extra supervision is not mixed into the comparison.

The comparison uses the same outer trials and channel subsets. Search budgets differ by
computational cost and are recorded, rather than claiming equal compute or exhaustive tuning.

## Evaluation contract

For each subject and CV seed, save the outer folds once and reuse them for every pipeline
and channel count. Stratify by **session × class** when counts allow, otherwise by class.
This is a mixed-session, within-subject evaluation of new trials. It does not estimate
performance on an unseen subject or an unseen recording session.

Inside each outer-training set, inner CV chooses the candidate using balanced accuracy of
concatenated inner OOF predictions, with log loss as tie-breaker. Neural early stopping
monitors inner-validation cross-entropy. The selected candidate is then retrained on **all
outer-training trials** for the median of its selected inner epoch counts. The learning-rate
schedule retains the same horizon as during tuning. Only then is the outer test fold scored.

All views of an original trial stay together. Multi-view neural training computes one loss
per original trial. Frozen pretrained features can be cached for all trials because the
encoder is fixed, in evaluation mode, and learns no subject statistics; probe preprocessing
is still fitted inside each training split.

Final subject metrics come from exact OOF predictions, not averages of fold accuracies.
Repeated CV seeds are averaged within subjects; subjects then receive equal weight.
Bootstrap intervals resample subjects and are descriptive across-subject uncertainty,
not evidence from treating folds or trials as independent participants.

Do not choose settings, seeds, or participants by repeatedly inspecting outer scores.
Any later development informed by these results is exploratory until independently checked.

## Configuration and reproducibility

In [ ]:
from __future__ import annotations
import os
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
import copy
import hashlib
import importlib.metadata
import itertools
import json
import math
import random
import re
import time
import warnings
from collections import OrderedDict
from dataclasses import dataclass, asdict, replace
from fractions import Fraction
from pathlib import Path

import cloudpickle
import numpy as np
import pandas as pd
from scipy import signal, stats
from scipy.special import softmax
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.covariance import oas
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.feature_selection import SelectKBest, f_classif, VarianceThreshold
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, f1_score,
                             confusion_matrix, log_loss, classification_report)
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from threadpoolctl import threadpool_limits
from tqdm.auto import tqdm

VERSION = "insubject-eight-window-cuda-1.0"
SOURCE_SHA256 = "development"  # replaced by the notebook builder
CLASSES = np.arange(4)
CLASS_NAMES = ("Up", "Down", "Right", "Left")
BANDS = ((2., 4.), (4., 8.), (8., 13.), (13., 20.), (20., 30.), (30., 45.))
WINDOWS = {"full": ((1., 3.5),), "late": ((1.5, 3.5),),
           "early": ((1., 3.),), "two_views": ((1., 3.), (1.5, 3.5))}


@dataclass
class ExperimentConfig:
    data_root: str = "/kaggle/input/datasets/truthisneverlinear/inner-speech-recognition/inner-speech-recognition/derivatives"
    output_root: str = "/kaggle/working/inner_speech_optimized"
    profile: str = "smoke"
    subject_ids: tuple | None = (1,)
    channel_counts: tuple = (128,)
    pipelines: tuple = ("classical", "eegnet", "labram")
    cv_seeds: tuple = (42,)
    outer_folds: int = 2
    inner_folds: int = 2
    ensemble_seeds: tuple = (0,)
    max_epochs: int = 3
    min_epochs: int = 1
    patience: int = 2
    scratch_batch_size: int = 32
    labram_batch_size: int = 8
    amp: bool = True
    device: str = "auto"
    cpu_threads: int = 2
    resume: bool = True
    expected_sfreq: float = 256.
    labram_repo: str = "braindecode/labram-pretrained"
    labram_revision: str = "0563b6c626e7b40d9a36653b763715db94d945d7"
    labram_offline: bool = False
    label_permutation_seed: int | None = None  # diagnostic only; labeled in every output
    # Optional explicit candidate lists. None selects the profile's documented list.
    svm_backend: str = "cpu"  # notebook default is CUDA; explicit CPU for local tests
    svm_calibration_folds: int = 3
    svm_cache_size: float = 512.
    svm_tol: float = 1e-3
    classical_feature_set: str = "combined"  # baseline, eight_window, or combined
    classical_candidates: tuple | None = None
    eegnet_candidates: tuple | None = None
    labram_candidates: tuple | None = None

    def validate(self):
        if self.profile not in ("smoke", "pilot", "full"):
            raise ValueError("profile must be smoke, pilot, or full")
        for field in ("channel_counts", "pipelines", "cv_seeds", "ensemble_seeds"):
            value = getattr(self, field)
            if not isinstance(value, (tuple, list)) or not value or len(set(value)) != len(value):
                raise ValueError(f"{field} must be a nonempty tuple with unique entries; use (128,), not (128)")
        if not set(self.channel_counts) <= {16, 32, 64, 128}:
            raise ValueError("Use channel counts 128, 64, 32, or 16")
        if not set(self.pipelines) <= {"classical", "eegnet", "labram"}:
            raise ValueError("Unknown pipeline")
        if self.subject_ids is not None and (not isinstance(self.subject_ids, (tuple, list)) or not self.subject_ids):
            raise ValueError("subject_ids must be None or a nonempty tuple")
        if min(self.outer_folds, self.inner_folds) < 2:
            raise ValueError("Both outer and inner CV need at least two folds")
        if not 1 <= self.min_epochs <= self.max_epochs or self.patience < 1:
            raise ValueError("Require 1 <= min_epochs <= max_epochs and patience >= 1")
        if self.cpu_threads < 1 or min(self.scratch_batch_size, self.labram_batch_size) < 1:
            raise ValueError("Thread count and batch sizes must be positive")
        if self.svm_backend not in ("cpu", "cuda"):
            raise ValueError("svm_backend must be cpu or cuda; no silent CPU fallback")
        if self.svm_calibration_folds < 2 or self.svm_cache_size <= 0 or self.svm_tol <= 0:
            raise ValueError("Require at least two SVM calibration folds and positive cache/tolerance")
        if self.classical_feature_set not in ("baseline", "eight_window", "combined"):
            raise ValueError("classical_feature_set must be baseline, eight_window, or combined")
        for pipeline in self.pipelines:
            candidate_list(self, pipeline)
        return self


def make_config(profile="smoke", **overrides):
    presets = {
        "smoke": dict(subject_ids=(1,), outer_folds=2, inner_folds=2,
                      max_epochs=3, min_epochs=1, patience=2, ensemble_seeds=(0,)),
        "pilot": dict(subject_ids=(1,), outer_folds=5, inner_folds=3,
                      max_epochs=60, min_epochs=15, patience=12, ensemble_seeds=(0,)),
        "full": dict(subject_ids=None, outer_folds=5, inner_folds=3,
                     max_epochs=100, min_epochs=20, patience=15, ensemble_seeds=(0, 1, 2)),
    }
    if profile not in presets:
        raise ValueError(f"Unknown profile: {profile}")
    return ExperimentConfig(profile=profile, **(presets[profile] | overrides)).validate()


def serializable(value):
    if isinstance(value, (Path, np.dtype)):
        return str(value)
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, np.generic):
        return value.item()
    raise TypeError(f"Cannot serialize {type(value)}")


def digest(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, default=serializable).encode()).hexdigest()[:24]


def file_sha(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for block in iter(lambda: f.read(8 * 1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()


def relative_path(path, directory):
    # Resolve macOS /var -> /private/var aliases before computing a portable path.
    return os.path.relpath(Path(path).resolve(), Path(directory).resolve())


def save_json(value, path):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_suffix(path.suffix + ".tmp")
    temp.write_text(json.dumps(value, indent=2, sort_keys=True, default=serializable) + "\n")
    temp.replace(path)


def save_csv(frame, path):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_suffix(path.suffix + ".tmp")
    frame.to_csv(temp, index=False); temp.replace(path)


def save_pickle(value, path):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_suffix(path.suffix + ".tmp")
    with temp.open("wb") as f:
        cloudpickle.dump(value, f)
    temp.replace(path)


def read_pickle(path):
    # Only load checkpoint/event pickle files from sources you trust.
    with Path(path).open("rb") as f:
        return cloudpickle.load(f)


def package_versions():
    result = {}
    for name in ("numpy", "scipy", "pandas", "scikit-learn", "mne", "torch", "braindecode", "pyriemann", "cloudpickle", "PyWavelets", "cuml-cu12", "cuml-cu13", "cupy-cuda12x", "cupy-cuda13x"):
        try:
            result[name] = importlib.metadata.version(name)
        except importlib.metadata.PackageNotFoundError:
            result[name] = None
    return result


def seed_everything(seed):
    random.seed(seed); np.random.seed(seed)
    import torch
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    torch.use_deterministic_algorithms(True, warn_only=True)


def metrics(y, prediction, probability=None):
    out = dict(accuracy=float(accuracy_score(y, prediction)),
               balanced_accuracy=float(balanced_accuracy_score(y, prediction)),
               macro_f1=float(f1_score(y, prediction, labels=CLASSES, average="macro", zero_division=0)))
    if probability is not None:
        probability = np.asarray(probability, np.float64)
        if not np.isfinite(probability).all() or (probability < 0).any() or not np.allclose(probability.sum(1), 1., atol=1e-5):
            raise ValueError("Invalid class probabilities")
        probability = probability / probability.sum(1, keepdims=True)
        out["log_loss"] = float(log_loss(y, probability, labels=CLASSES))
    return out

## Audited data loading and identical trial splits

The loader uses the standardized event mapping (`condition==1`, labels 0..3), accounts for
dropped epochs, and selects electrodes by BioSemi names rather than numeric position.
It converts volts to microvolts exactly once. Source FIF/event hashes, trial IDs, amplitude
checks, and channel order are saved. The processed derivatives are assumed to carry the
provider's original preprocessing; header values alone do not prove ICA history.

No additional VMD, ICA, or label-driven trial rejection is applied. Each original trial is
included in evaluation. Added filters operate on the action interval only, so they do not
introduce additional visual-cue samples through filtering. Existing derivative preprocessing
may already have temporal effects; action-window evaluation does not prove neural specificity.

In [ ]:
def discover_recordings(root):
    records = []
    for path in sorted(Path(root).rglob("*_eeg-epo.fif")):
        match = re.search(r"sub-(\d+)_ses-(\d+)_eeg-epo\.fif$", path.name)
        if not match:
            continue
        subject, session = map(int, match.groups())
        events = path.with_name(path.name.replace("_eeg-epo.fif", "_events.dat"))
        if not events.exists():
            raise FileNotFoundError(events)
        records.append((subject, session, path, events))
    if not records:
        raise FileNotFoundError(f"No processed sub-XX_ses-XX_eeg-epo.fif files below {root}")
    if len({(s, r) for s, r, *_ in records}) != len(records):
        raise ValueError("Duplicate subject/session recordings: point DATA_ROOT at one dataset version")
    return records


def load_subject(cfg, subject):
    import mne
    names = mne.channels.make_standard_montage("biosemi128").ch_names
    blocks, rows, provenance = [], [], []
    for sid, session, path, event_path in discover_recordings(cfg.data_root):
        if sid != subject:
            continue
        epochs = mne.read_epochs(path, preload=True, verbose=False)
        if not np.isclose(epochs.info["sfreq"], cfg.expected_sfreq):
            raise ValueError(f"{path.name}: expected processed {cfg.expected_sfreq} Hz data")
        events = pd.read_pickle(event_path)
        if isinstance(events, pd.DataFrame):
            columns = {str(k).lower(): k for k in events.columns}
            label = columns.get("code", columns.get("label"))
            condition = columns.get("condition")
            if label is None or condition is None:
                raise ValueError(f"Unrecognized event columns: {events.columns.tolist()}")
            labels, conditions = events[label].to_numpy(), events[condition].to_numpy()
        else:
            array = np.asarray(events)
            if array.ndim != 2 or array.shape[1] < 4:
                raise ValueError(f"Invalid event matrix in {event_path}")
            labels, conditions = array[:, 1], array[:, 2]
        if len(labels) == len(epochs.drop_log):
            labels, conditions = labels[epochs.selection], conditions[epochs.selection]
        elif len(labels) != len(epochs):
            raise ValueError(f"Epoch/event count mismatch in {path.name}; no truncation is allowed")
        if not np.isin(labels, CLASSES).all() or not np.isin(conditions, (0, 1, 2)).all():
            raise ValueError("Expected standardized labels 0..3 and conditions 0..2")
        if set(names) - set(epochs.ch_names):
            raise ValueError("The recording does not contain the 128 BioSemi EEG electrodes")
        epochs.pick(names).reorder_channels(names)
        mask = (epochs.times >= 1. - 1e-9) & (epochs.times < 3.5 - 1e-9)
        if mask.sum() != round(2.5 * cfg.expected_sfreq):
            raise ValueError("The FIF time axis does not contain the expected [1.0,3.5) action interval")
        selected = np.flatnonzero(conditions == 1)
        # Only action samples enter our added filters, preventing extra pre-cue smearing.
        X = epochs.get_data(copy=True)[selected][:, :, mask].astype(np.float32) * 1e6
        if not np.isfinite(X).all():
            raise ValueError("Nonfinite EEG; inspect the source rather than silently imputing waveforms")
        blocks.append(X)
        for index in selected:
            rows.append(dict(subject=sid, session=session, y_true=int(labels[index]),
                             trial_id=f"sub-{sid:02d}_ses-{session:02d}_row-{int(epochs.selection[index]):04d}"))
        provenance.append(dict(eeg_name=path.name, eeg_sha256=file_sha(path),
                               events_name=event_path.name, events_sha256=file_sha(event_path),
                               trials=len(selected), sfreq=float(epochs.info["sfreq"]),
                               highpass=float(epochs.info["highpass"]), lowpass=float(epochs.info["lowpass"])))
    if not blocks:
        raise ValueError(f"No trials for subject {subject}")
    meta = pd.DataFrame(rows)
    if meta.trial_id.duplicated().any() or set(meta.y_true) != set(CLASSES):
        raise ValueError("Duplicate trial IDs or missing classes")
    X = np.concatenate(blocks)
    amplitude = np.std(X, axis=-1)
    print(f"Subject {subject:02d}: {len(meta)} trials; counts {meta.y_true.value_counts().sort_index().to_dict()}; "
          f"median channel SD {np.median(amplitude):.2f} microvolts")
    audit = dict(provenance=provenance, channel_order=names, units="microvolts", window=[1., 3.5],
                 shape=list(X.shape), channel_sd_percentiles=np.percentile(amplitude, [0, 1, 50, 99, 100]).tolist(),
                 no_extra_vmd_ica_or_trial_rejection=True)
    return X, names, meta, audit


def channel_subsets(names):
    import mne
    positions = mne.channels.make_standard_montage("biosemi128").get_positions()["ch_pos"]
    xyz = np.array([positions[name] for name in names])
    chosen = [int(np.argmax(xyz[:, 2]))]
    while len(chosen) < len(names):
        distances = np.linalg.norm(xyz[:, None] - xyz[chosen][None], axis=-1).min(axis=1)
        distances[chosen] = -np.inf
        chosen.append(int(np.argmax(distances)))
    return {128: np.arange(len(names)), **{n: np.asarray(chosen[:n]) for n in (64, 32, 16)}}


def stratified_splits(y, sessions, n_folds, seed):
    y, sessions = np.asarray(y), np.asarray(sessions)
    if set(y) != set(CLASSES) or np.bincount(y, minlength=4).min() < n_folds:
        raise ValueError(f"Insufficient trials per class for {n_folds} folds")
    joint = np.asarray([f"{s}_{c}" for s, c in zip(sessions, y)])
    counts = pd.Series(joint).value_counts()
    stratify = joint if counts.min() >= n_folds else y
    mode = "session_and_class" if counts.min() >= n_folds else "class_only"
    splits = list(StratifiedKFold(n_folds, shuffle=True, random_state=seed).split(y, stratify))
    coverage = np.zeros(len(y), int)
    for train, test in splits:
        if set(train) & set(test):
            raise AssertionError("Overlapping train/test trials")
        coverage[test] += 1
    if not np.all(coverage == 1):
        raise AssertionError("CV must predict every original trial exactly once")
    return splits, mode


def make_split_plan(meta, cfg, seed):
    outer, mode = stratified_splits(meta.y_true, meta.session, cfg.outer_folds, seed)
    plans = []
    for fold, (train, test) in enumerate(outer, 1):
        inner, inner_mode = stratified_splits(meta.y_true.iloc[train], meta.session.iloc[train],
                                             cfg.inner_folds, seed + fold * 1009)
        plans.append(dict(fold=fold, train=train.tolist(), test=test.tolist(),
                          inner=[dict(train=train[a].tolist(), validation=train[b].tolist()) for a, b in inner],
                          stratification=mode, inner_stratification=inner_mode))
    return dict(seed=int(seed), trial_ids=meta.trial_id.tolist(), labels=meta.y_true.tolist(), folds=plans)

## Deterministic signal views and compact classical representations

The six filter bands are 2–4, 4–8, 8–13, 13–20, 20–30, and 30–45 Hz. Fixed fourth-order
Butterworth filters run separately on each trial. Their coefficients use no dataset estimates.
Bandpower features retain both the whole-window power and three ordered temporal bins.
Spatial PCA and all feature selection are fitted again inside each training split.

Riemannian candidates use OAS covariance shrinkage and an affine-invariant tangent-space
reference learned from training trials only (`tsupdate=False`). A compact PCA representation
then regularizes the classifier. This is not an all-electrode connectivity interpretation.

Window choices are `full=[1.0,3.5)`, `late=[1.5,3.5)`, and, for selected neural candidates,
`two_views=[1.0,3.0)` plus `[1.5,3.5)`. All remain within the action interval.

In [ ]:
def resample(data, source_sfreq, target_sfreq):
    if np.isclose(source_sfreq, target_sfreq):
        return np.asarray(data, np.float32)
    ratio = Fraction(float(target_sfreq) / float(source_sfreq)).limit_denominator(10000)
    return signal.resample_poly(data, ratio.numerator, ratio.denominator, axis=-1).astype(np.float32)


def bandpass(data, sfreq, band):
    sos = signal.butter(4, band, btype="bandpass", fs=sfreq, output="sos")
    return signal.sosfiltfilt(sos, data, axis=-1).astype(np.float32)


def cut_views(data, sfreq, view):
    result = [data[..., round((start-1.)*sfreq):round((stop-1.)*sfreq)] for start, stop in WINDOWS[view]]
    return np.stack(result, axis=1)


class SubjectViews:
    """Fixed per-trial transforms only. No labels or fitted statistics here."""
    def __init__(self, X, sfreq):
        self.X, self.sfreq = np.asarray(X, np.float32), sfreq
        self.cache = OrderedDict()

    def get(self, family, view):
        key = (family, view)
        if key in self.cache:
            self.cache.move_to_end(key)
            return self.cache[key]
        if family in ("eight_window", "bandpower_eight_window"):
            broadband = bandpass(self.X, self.sfreq, (2., 45.))
            signals = [broadband]
            if family == "bandpower_eight_window":
                signals += [bandpass(self.X, self.sfreq, band) for band in BANDS]
            # Preserve the original 256-Hz bandpower; extractor resamples broadband to 128 Hz.
            output = cut_views(np.stack(signals, axis=1), self.sfreq, view)
        elif family in ("classical", "fbcnet"):
            rate = self.sfreq if family == "classical" else 128.
            data = np.stack([resample(bandpass(self.X, self.sfreq, band), self.sfreq, rate) for band in BANDS], axis=1)
            # [trial, view, band, electrode, time]
            output = cut_views(data, rate, view)
        elif family == "eegnet":
            data = resample(bandpass(self.X, self.sfreq, (2., 45.)), self.sfreq, 128.)
            output = cut_views(data, 128., view)
        elif family == "labram":
            # Preserve the pretrained amplitude convention: microvolts / 100.
            data = resample(self.X, self.sfreq, 200.) / 100.
            output = cut_views(data, 200., view)
            padding = (-output.shape[-1]) % 200
            if padding:
                pads = [(0, 0)] * output.ndim
                pads[-1] = (0, padding)  # right-padding preserves the action-start alignment
                output = np.pad(output, pads)
        else:
            raise ValueError(f"Unknown signal family {family}")
        if len(self.cache) >= 2:
            self.cache.popitem(last=False)
        self.cache[key] = np.ascontiguousarray(output, np.float32)
        return self.cache[key]


class SpatialFeatures(BaseEstimator, TransformerMixin):
    """Train-fitted spatial PCA, then multiclass CSP or Riemannian tangent mapping.

    Input: trials x bands x electrodes x samples (one temporal view).
    OAS shrinks covariance matrices, including when average referencing reduced rank.
    """
    def __init__(self, representation="bandpower", rank=16, components=4, bins=3):
        self.representation, self.rank = representation, rank
        self.components, self.bins = components, bins

    def fit(self, X, y):
        self.n_features_in_ = X.shape[2]
        self.models_ = []
        if self.representation == "bandpower":
            return self
        for band in range(X.shape[1]):
            data = np.asarray(X[:, band], np.float64)
            centered = data - data.mean(axis=-1, keepdims=True)
            cov = np.einsum("nct,ndt->cd", centered, centered) / (len(data) * data.shape[-1])
            _, vectors = np.linalg.eigh(cov)
            rank = min(self.rank, data.shape[1] - 1)
            projection = vectors[:, -rank:]
            projected = np.einsum("cr,nct->nrt", projection, centered)
            if self.representation == "csp":
                from mne.decoding import CSP
                model = CSP(n_components=min(self.components, rank), reg="oas", log=True,
                            cov_est="concat", norm_trace=False, rank="full")
                import mne
                with mne.utils.use_log_level("ERROR"):
                    model.fit(projected, y)
            elif self.representation == "riemann":
                from pyriemann.tangentspace import TangentSpace
                model = TangentSpace(metric="riemann", tsupdate=False).fit(self._covariances(projected))
            else:
                raise ValueError(self.representation)
            self.models_.append((projection, model))
        return self

    @staticmethod
    def _covariances(X):
        return np.stack([oas(trial.T, assume_centered=True)[0] for trial in X])

    def transform(self, X):
        if self.representation == "bandpower":
            slices = [slice(None)] + [slice(int(a), int(b)) for a, b in zip(
                np.linspace(0, X.shape[-1], self.bins+1)[:-1], np.linspace(0, X.shape[-1], self.bins+1)[1:])]
            return np.concatenate([np.log(np.maximum(np.mean(X[..., s]**2, axis=-1), 1e-12)).reshape(len(X), -1)
                                   for s in slices], axis=1)
        features = []
        for band, (projection, model) in enumerate(self.models_):
            centered = np.asarray(X[:, band], np.float64)
            centered = centered - centered.mean(axis=-1, keepdims=True)
            projected = np.einsum("cr,nct->nrt", projection, centered)
            features.append(model.transform(projected if self.representation == "csp" else self._covariances(projected)))
        return np.concatenate(features, axis=1)


class CompactFeatures(BaseEstimator, TransformerMixin):
    """Dimension limits adapt to the available training samples only."""
    def __init__(self, dimension=32, method="anova"):
        self.dimension, self.method = dimension, method

    def fit(self, X, y):
        self.variance_ = VarianceThreshold(1e-12).fit(X)
        X = self.variance_.transform(X)
        self.scaler_ = StandardScaler().fit(X)
        X = self.scaler_.transform(X)
        dimension = min(self.dimension, X.shape[1], max(1, len(X)-len(CLASSES)))
        self.reducer_ = (PCA(n_components=dimension, svd_solver="full") if self.method == "pca"
                         else SelectKBest(f_classif, k=dimension))
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", RuntimeWarning)
            self.reducer_.fit(X, y)
        return self

    def transform(self, X):
        return self.reducer_.transform(self.scaler_.transform(self.variance_.transform(X)))

## Eight-window AR, wavelet entropy and wavelet variance

Every original trial produces **one feature row**, with all eight windows kept together.
This is a clearly specified adaptation of the paper's feature idea, rather than its ambiguous
120/170-feature construction. All three methods are applied to **every window and electrode**.

1. Use our action-only microvolt EEG, filtered to 2-45 Hz. Resample the broadband signal
   from 256 to 128 Hz with polyphase anti-aliasing.
2. Split the full or late view into eight non-overlapping, contiguous windows. Full trials
   contain 320 samples: eight windows of 40 samples (312.5 ms). Late trials contain
   256 samples: eight windows of 32 samples (250 ms).
3. Estimate univariate Burg AR coefficients after centering each window. Pilot/smoke use
   order 4; full also tests order 1. No labels or cross-trial statistics enter this estimation.
4. Apply `db2`, **three-level decimated DWT**, with symmetric boundary extension. Retain
   A3, D3, D2, D1 coefficient sets. At 128 Hz their nominal ranges are 0-8, 8-16,
   16-32, 32-64 Hz; they are approximate overlapping filter responses, with input already
   limited to 2-45 Hz. This is not the paper's undecimated wavelet-packet entropy method.
5. Extract normalized Shannon entropy and log population variance from each coefficient set.
   Entropy uses normalized squared coefficients, the conventional minus sign, and division
   by log(coefficient count). Zero-energy components receive entropy zero. Boundary
   coefficients are retained and this choice is recorded.
6. Flatten windows, electrodes and descriptors into one row. Optionally append our original
   six-band, whole-window/three-bin bandpower features, computed at the original 256 Hz.
7. Fit constant-feature removal, standardization and ANOVA selection using training trials
   only. Test compact representations with linear/RBF SVM within nested CV.

With AR order 4, each electrode/window contributes 4 AR + 4 entropy + 4 log-variance
values: **12,288 raw columns at 128 electrodes**, or **15,360 with bandpower fusion**.
These are reduced to the candidate dimension (usually 32) before classification. They are
not additional independent training samples. Short windows cannot provide precise estimates
of slow rhythms; increased temporal detail may or may not improve performance.

`CLASSICAL_FEATURE_SET="combined"` compares the original representations and both new
ones. `"eight_window"` runs only the new standalone/fused representations; `"baseline"`
runs the original representations. With combined defaults, classical candidate counts are
5 / 26 / 92 for smoke / pilot / full. All candidates share the original trial splits.

Scratch EEGNet/filter-bank CNN and LaBraM remain available using their existing EEG inputs.
The default switches enable classical only for this focused feature experiment.

Sources: [paper](https://doi.org/10.1109/ACCESS.2024.3474854),
[PyWavelets](https://pywavelets.readthedocs.io/en/latest/ref/dwt-discrete-wavelet-transform.html),
[Burg AR estimation](https://www.statsmodels.org/stable/generated/statsmodels.regression.linear_model.burg.html).

In [ ]:
"""Fixed per-trial descriptors: eight windows of Burg AR, wavelet entropy and variance.

This is an explicit adaptation of the paper's idea, not a reconstruction of its
ambiguous 120/170-column recipe. All eight windows form one row per original trial.
"""
import numpy as np
import pywt
from scipy import signal
from fractions import Fraction
from sklearn.base import BaseEstimator, TransformerMixin


def burg_coefficients(data, order=4):
    """Vectorized univariate Burg estimates; returns positive prediction coefficients.

Each row is estimated independently, including test rows. There is no fitting of
population statistics or use of labels here. Constant signals return zero.
"""
    x = np.asarray(data, dtype=np.float64)
    if x.ndim != 2 or not np.isfinite(x).all():
        raise ValueError("Burg input must be finite [signals, samples]")
    if not isinstance(order, (int, np.integer)) or not 1 <= order < x.shape[1] - 1:
        raise ValueError("AR order must be positive and smaller than samples minus one")
    x = x - x.mean(axis=-1, keepdims=True)
    scale = np.sqrt(np.mean(x*x, axis=-1, keepdims=True))
    x = x / np.maximum(scale, 1e-15)
    forward, backward = x[:, 1:].copy(), x[:, :-1].copy()
    coefficients = np.empty((len(x), 0), dtype=np.float64)
    for lag in range(order):
        denominator = np.sum(forward*forward + backward*backward, axis=-1)
        reflection = np.divide(-2*np.sum(forward*backward, axis=-1), denominator,
                               out=np.zeros(len(x)), where=denominator > 1e-15)
        reflection = np.clip(reflection, -1 + 1e-7, 1 - 1e-7)
        previous = coefficients
        coefficients = np.concatenate([
            previous + reflection[:, None]*previous[:, ::-1], reflection[:, None]], axis=1)
        updated_forward = forward + reflection[:, None]*backward
        updated_backward = backward + reflection[:, None]*forward
        forward, backward = updated_forward[:, 1:], updated_backward[:, :-1]
    return -coefficients


def wavelet_statistics(coefficients):
    """Normalized coefficient-energy entropy and log population variance.

Entropy is -sum(p log p)/log(number of coefficients). It is zero for
zero-energy or one-coefficient components. Boundary coefficients are retained;
the symmetric extension mode and this choice are recorded in every checkpoint.
"""
    values = np.asarray(coefficients, dtype=np.float64)
    energy = values*values
    total = energy.sum(axis=-1, keepdims=True)
    probability = np.divide(energy, total, out=np.zeros_like(energy), where=total > 1e-24)
    entropy = -np.sum(probability*np.log(np.maximum(probability, 1e-300)), axis=-1)
    entropy = entropy/np.log(values.shape[-1]) if values.shape[-1] > 1 else np.zeros_like(entropy)
    variance = np.log(np.maximum(np.var(values, axis=-1, ddof=0), 1e-12))
    return np.clip(entropy, 0., 1.), variance


class EightWindowFeatures(BaseEstimator, TransformerMixin):
    """Eight contiguous windows; all channels/descriptors concatenate into ONE trial row.

Input slots are broadband 2-45 Hz and, for fusion, the existing six filter bands.
Broadband input is resampled from 256 to 128 Hz for compact wavelet/AR analysis.
Bandpower fusion uses the six 256-Hz bands, exactly as the original representation.
No fitted spatial projection, normalization, imputation or label use occurs here.
"""
    def __init__(self, representation="eight_window", n_windows=8, wavelet="db2",
                 wavelet_level=3, ar_order=4, input_sfreq=256., feature_sfreq=128.):
        self.representation = representation
        self.n_windows = n_windows
        self.wavelet = wavelet
        self.wavelet_level = wavelet_level
        self.ar_order = ar_order
        self.input_sfreq = input_sfreq
        self.feature_sfreq = feature_sfreq

    def _validate(self, X):
        X = np.asarray(X)
        slots = 7 if self.representation == "bandpower_eight_window" else 1
        if self.representation not in ("eight_window", "bandpower_eight_window"):
            raise ValueError("Unknown eight-window representation")
        if X.ndim != 4 or X.shape[1] != slots or min(X.shape) < 1 or not np.isfinite(X).all():
            raise ValueError(f"Expected finite [trials, {slots} signal slots, channels, samples]")
        if self.n_windows != 8:
            raise ValueError("This experiment requires exactly eight windows")
        if self.input_sfreq <= 0 or self.feature_sfreq <= 0:
            raise ValueError("Sampling frequencies must be positive")
        ratio = Fraction(float(self.feature_sfreq)/float(self.input_sfreq)).limit_denominator(10000)
        samples = (X.shape[-1]*ratio.numerator + ratio.denominator - 1)//ratio.denominator
        minimum_window = samples//self.n_windows
        maximum = pywt.dwt_max_level(minimum_window, self.wavelet)
        if not isinstance(self.wavelet_level, (int, np.integer)) or not 1 <= self.wavelet_level <= maximum:
            raise ValueError(f"Wavelet level {self.wavelet_level} is too deep for {minimum_window}-sample windows; maximum {maximum}")
        if not isinstance(self.ar_order, (int, np.integer)) or not 1 <= self.ar_order < minimum_window - 1:
            raise ValueError("AR order is too large for the window")
        return X, ratio, samples

    def fit(self, X, y=None):
        X, _, samples = self._validate(X)
        self.n_channels_ = X.shape[2]
        self.input_samples_ = X.shape[-1]
        self.window_lengths_ = [len(window) for window in np.array_split(np.arange(samples), self.n_windows)]
        self.component_labels_ = [f"A{self.wavelet_level}"] + [f"D{level}" for level in range(self.wavelet_level, 0, -1)]
        self.n_features_out_ = self.n_windows*self.n_channels_*(self.ar_order + 2*len(self.component_labels_))
        if self.representation == "bandpower_eight_window":
            self.n_features_out_ += 4*6*self.n_channels_
        return self

    def transform(self, X):
        X, ratio, _ = self._validate(X)
        if not hasattr(self, "n_channels_"):
            raise ValueError("Fit the extractor before transform")
        if X.shape[2:] != (self.n_channels_, self.input_samples_):
            raise ValueError("Channel count or window length differs from the fitted extractor")
        broadband = signal.resample_poly(X[:, 0], ratio.numerator, ratio.denominator, axis=-1)
        windows = np.array_split(broadband, self.n_windows, axis=-1)
        all_windows = []
        for window in windows:
            rows = window.reshape(-1, window.shape[-1])
            ar = burg_coefficients(rows, self.ar_order)
            coefficients = pywt.wavedec(rows.astype(np.float64), self.wavelet,
                                       mode="symmetric", level=self.wavelet_level, axis=-1)
            summaries = [wavelet_statistics(component) for component in coefficients]
            entropy = np.stack([item[0] for item in summaries], axis=-1)
            variance = np.stack([item[1] for item in summaries], axis=-1)
            descriptors = np.concatenate([ar, entropy, variance], axis=-1)
            all_windows.append(descriptors.reshape(len(X), -1))
        features = np.concatenate(all_windows, axis=1)
        if self.representation == "bandpower_eight_window":
            edges = np.linspace(0, X.shape[-1], 4).astype(int)
            slices = [slice(None)] + [slice(a, b) for a, b in zip(edges[:-1], edges[1:])]
            power = [np.log(np.maximum(np.mean(X[:, 1:, :, section]**2, axis=-1), 1e-12)).reshape(len(X), -1)
                     for section in slices]
            features = np.concatenate([features, *power], axis=1)
        if features.shape != (len(X), self.n_features_out_) or not np.isfinite(features).all():
            raise ValueError("Invalid eight-window feature matrix")
        return features.astype(np.float32)

    def describe(self, channel_names=None):
        """Map every raw feature column to its electrode, window and descriptor."""
        import pandas as pd
        names = list(channel_names) if channel_names is not None else [f"electrode_{i:03d}" for i in range(self.n_channels_)]
        if len(names) != self.n_channels_ or len(set(names)) != len(names):
            raise ValueError("Feature names require the fitted number of unique channels")
        rows = []
        descriptors = [("ar", f"lag{lag}") for lag in range(1, self.ar_order+1)]
        descriptors += [("wavelet_entropy", label) for label in self.component_labels_]
        descriptors += [("wavelet_log_variance", label) for label in self.component_labels_]
        offset = 0
        for window, length in enumerate(self.window_lengths_, 1):
            for channel in names:
                for family, component in descriptors:
                    rows.append(dict(feature_name=f"window{window:02d}__{channel}__{family}__{component}",
                                     family=family, channel=channel, window=window, component=component,
                                     start_offset_s=offset/self.feature_sfreq,
                                     stop_offset_s=(offset+length)/self.feature_sfreq))
            offset += length
        if self.representation == "bandpower_eight_window":
            edges = np.linspace(0, self.input_samples_, 4).astype(int)
            bounds = [(0, self.input_samples_)] + list(zip(edges[:-1], edges[1:]))
            bands = ("2-4Hz", "4-8Hz", "8-13Hz", "13-20Hz", "20-30Hz", "30-45Hz")
            for temporal, (start, stop) in enumerate(bounds):
                for band in bands:
                    for channel in names:
                        rows.append(dict(feature_name=f"bandpower{temporal:02d}__{channel}__{band}",
                                         family="bandpower", channel=channel, window=temporal, component=band,
                                         start_offset_s=start/self.input_sfreq, stop_offset_s=stop/self.input_sfreq))
        frame = pd.DataFrame(rows)
        frame.insert(0, "input_column", np.arange(len(frame)))
        if len(frame) != self.n_features_out_ or frame.feature_name.duplicated().any():
            raise AssertionError("Feature schema does not match the extracted columns")
        return frame

    def get_feature_names_out(self, input_features=None):
        return self.describe(input_features).feature_name.to_numpy()

    def metadata(self):
        return dict(representation=self.representation, n_windows=self.n_windows,
                    wavelet=self.wavelet, wavelet_level=self.wavelet_level,
                    wavelet_mode="symmetric", boundary_coefficients="retained",
                    ar_method="univariate_burg_demeaned", ar_order=self.ar_order,
                    entropy="negative_sum_p_log_p_divided_by_log_coefficient_count",
                    variance="log_population_variance_floor_1e-12", input_units="microvolts",
                    input_sfreq=self.input_sfreq, feature_sfreq=self.feature_sfreq,
                    window_lengths=self.window_lengths_, window_lengths_s=[n/self.feature_sfreq for n in self.window_lengths_],
                    wavelet_components=self.component_labels_, raw_feature_count=self.n_features_out_,
                    paper_exact_reproduction=False, one_row_per_original_trial=True)

## CUDA SVM backend and portable checkpoints

SVM training defaults to **CUDA using cuML**. Four-class fitting uses six explicit binary
linear/RBF SVMs; it never calls cuML's sklearn multiclass wrapper. Inputs are CuPy float32
arrays. Native margins and support vectors are converted explicitly before NumPy aggregation,
avoiding the mixed NumPy/CuPy multiclass error.

Binary probabilities use three-fold, classifier-only OOF sigmoid calibration wholly within
the current training split. Pairwise probabilities use Wu-Lin-Weng coupling. This differs
from sklearn SVC's built-in five-fold probability calibration; backend, calibration recipe
and hardware are saved. Calibration adds fits because reports retain probability/log-loss
evaluation. The feature transformations are fitted on the current training split before
this classifier-only calibration; outer test data are never used.

Support vectors, coefficients and calibration are saved as NumPy arrays. Each export is
checked against the native binary decision function. Saved models make portable CPU
predictions without requiring cuML/CUDA. GPU acceleration applies to **SVM fitting**;
feature extraction, selection, sigmoid calibration/coupling, portable prediction and LDA
remain CPU operations.

The runner performs an actual four-class linear/RBF CUDA preflight before EEG processing.
It fails if CUDA cannot be used: no silent CPU fallback. One visible GPU is used. Small
subject SVMs may not be faster because launch/transfer overhead can dominate; CPU feature
extraction can still limit total runtime.

Sources: [cuML SVC](https://docs.nvidia.com/cuml/latest/api/generated/cuml.svm.SVC/),
[RAPIDS installation](https://docs.nvidia.com/datascience/install/),
[LIBSVM probability coupling](https://github.com/cjlin1/libsvm/blob/master/svm.cpp).

In [ ]:
"""CUDA binary SVM fitting, explicit OVO probability coupling, portable inference.

cuML is imported only for fitting on CUDA. Saved estimators contain NumPy arrays,
not CUDA objects. CPU mode uses the same explicit algorithm for backend tests.
"""
import itertools
import numpy as np
from scipy.optimize import minimize
from scipy.special import expit
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.model_selection import StratifiedKFold
from sklearn.svm import SVC


def cuda_svm_libraries():
    """Require a functioning cuML/CuPy CUDA environment; never switch to CPU silently."""
    try:
        import cupy as cp
        import cuml
        from cuml.svm import SVC as CuSVC
        if cp.cuda.runtime.getDeviceCount() < 1:
            raise RuntimeError("No visible CUDA device")
        device = int(cp.cuda.runtime.getDevice())
        properties = cp.cuda.runtime.getDeviceProperties(device)
        major, minor = int(properties["major"]), int(properties["minor"])
        if (major, minor) < (7, 0):
            raise RuntimeError("This RAPIDS setup requires compute capability 7.0 or later")
        name = properties["name"]
        name = name.decode() if isinstance(name, bytes) else str(name)
        cp.asarray([1., 2.], dtype=cp.float32).sum().get()
        cp.cuda.runtime.deviceSynchronize()
    except Exception as exc:
        raise RuntimeError(
            "CUDA SVM is required but cuML/CuPy could not use an NVIDIA GPU. "
            "Enable a Kaggle T4 and run the CUDA installation/preflight cells. "
            "Use SVM_BACKEND='cpu' only if you explicitly want CPU fitting. "
            f"Original error: {exc}") from exc
    return cp, CuSVC, dict(backend="cuda", device=device, name=name,
                           compute_capability=f"{major}.{minor}",
                           cupy_version=cp.__version__, cuml_version=cuml.__version__,
                           cuda_runtime=int(cp.cuda.runtime.runtimeGetVersion()),
                           cuda_driver=int(cp.cuda.runtime.driverGetVersion()))


def svm_host_array(value):
    """Explicit CUDA-to-host conversion before any NumPy aggregation."""
    if hasattr(value, "to_output"):
        value = value.to_output("numpy")
    elif hasattr(value, "get"):
        value = value.get()
    return np.asarray(value)


def svm_portable_margin(parameters, X):
    """Evaluate a binary linear/RBF support-vector model on CPU after restoration."""
    X = np.asarray(X, np.float64)
    support = parameters["support_vectors"]
    alpha = parameters["dual_coef"]
    if parameters["kernel"] == "linear":
        return X @ (alpha @ support) + parameters["intercept"]
    result = []
    support_norm = np.sum(support*support, axis=1)
    for start in range(0, len(X), 1024):
        block = X[start:start+1024]
        squared = np.maximum(np.sum(block*block, axis=1)[:, None] + support_norm[None] - 2*block@support.T, 0.)
        result.append(np.exp(-parameters["gamma"]*squared) @ alpha + parameters["intercept"])
    return np.concatenate(result) if result else np.empty(0)


def svm_export_binary(model, X_probe, native_margin, kernel, gamma):
    """Validate the portable representation against the native binary decision function."""
    parameters = dict(kernel=kernel, gamma=float(gamma),
                      support_vectors=svm_host_array(model.support_vectors_).astype(np.float64),
                      dual_coef=svm_host_array(model.dual_coef_).astype(np.float64).reshape(-1),
                      intercept=float(svm_host_array(model.intercept_).reshape(-1)[0]))
    native = svm_host_array(native_margin).astype(np.float64).reshape(-1)
    portable = svm_portable_margin(parameters, X_probe)
    direct_error = np.max(np.abs(portable-native), initial=0.)
    reverse_error = np.max(np.abs(-portable-native), initial=0.)
    if reverse_error < direct_error:
        parameters["dual_coef"] *= -1
        parameters["intercept"] *= -1
        portable = -portable
    if not np.allclose(portable, native, atol=2e-3, rtol=2e-3):
        raise RuntimeError("Exported SVM margins do not match native margins; stop rather than save an incompatible model")
    if not all(np.isfinite(parameters[key]).all() for key in ("support_vectors", "dual_coef")):
        raise ValueError("Nonfinite support-vector parameters")
    return parameters


def svm_sigmoid_fit(scores, labels):
    """Platt-style sigmoid on classifier-only OOF margins, with smoothed targets."""
    scores, labels = np.asarray(scores, np.float64), np.asarray(labels, int)
    if not np.isfinite(scores).all() or set(labels) != {0, 1}:
        raise ValueError("Calibration requires finite OOF margins and both classes")
    positive, negative = int(labels.sum()), int((labels == 0).sum())
    target = np.where(labels == 1, (positive+1.)/(positive+2.), 1./(negative+2.))
    mean, scale = float(scores.mean()), max(float(scores.std()), 1e-12)
    normalized = (scores-mean)/scale
    def objective(parameters):
        z = parameters[0]*normalized + parameters[1]
        error = expit(z)-target
        loss = np.mean(np.logaddexp(0., z)-target*z) + 1e-8*parameters[0]**2
        gradient = np.array([np.mean(error*normalized) + 2e-8*parameters[0], error.mean()])
        return loss, gradient
    optimum = minimize(objective, [0., np.log((positive+1.)/(negative+1.))],
                       method="L-BFGS-B", jac=True, options={"maxiter": 200, "ftol": 1e-12})
    if not optimum.success or not np.isfinite(optimum.x).all():
        raise RuntimeError(f"SVM sigmoid calibration failed: {optimum.message}")
    return dict(slope=float(optimum.x[0]), bias=float(optimum.x[1]), mean=mean, scale=scale)


def svm_pairwise_coupling(pairwise):
    """Wu-Lin-Weng coupling of binary probabilities, as used by LIBSVM.

pairwise[:,i,j] is the probability of class i conditional on i or j.
All operations are NumPy; no sklearn multiclass/CuPy mixture is involved.
"""
    pairwise = np.asarray(pairwise, np.float64)
    if pairwise.ndim != 3 or pairwise.shape[1] != pairwise.shape[2] or not np.isfinite(pairwise).all():
        raise ValueError("Expected finite [trials, classes, classes] pairwise probabilities")
    n_classes = pairwise.shape[1]
    if n_classes < 2:
        raise ValueError("At least two classes required")
    if n_classes == 2:
        return np.stack([pairwise[:, 0, 1], pairwise[:, 1, 0]], axis=1)
    result = []
    for matrix in pairwise:
        Q = -matrix.T*matrix
        np.fill_diagonal(Q, np.sum(matrix*matrix, axis=0))
        probability = np.full(n_classes, 1./n_classes)
        for iteration in range(max(100, n_classes)):
            Qp = Q @ probability
            pQp = float(probability @ Qp)
            if np.max(np.abs(Qp-pQp)) < .005/n_classes:
                break
            for index in range(n_classes):
                difference = (-Qp[index]+pQp)/Q[index, index]
                probability[index] += difference
                denominator = 1.+difference
                pQp = (pQp+difference*(difference*Q[index, index]+2*Qp[index]))/denominator**2
                Qp = (Qp+difference*Q[:, index])/denominator
                probability /= denominator
        else:
            raise RuntimeError("Pairwise probability coupling did not converge")
        probability = np.maximum(probability, 1e-12)
        result.append(probability/probability.sum())
    return np.asarray(result).reshape(-1, n_classes)


class PortableCudaSVC(ClassifierMixin, BaseEstimator):
    """Six binary CUDA SVMs for four classes; no cuML multiclass wrapper.

Fit uses cuML on CUDA. Calibration and aggregation run on CPU. Portable CPU
prediction avoids requiring GPU libraries to load saved checkpoints.
Three-fold classifier-only OOF calibration differs from sklearn's default
probability=True five-fold implementation; the method is saved explicitly.
"""
    def __init__(self, C=1., kernel="rbf", gamma="scale", backend="cuda",
                 calibration_folds=3, cache_size=512., tol=1e-3, random_state=42):
        self.C, self.kernel, self.gamma = C, kernel, gamma
        self.backend, self.calibration_folds = backend, calibration_folds
        self.cache_size, self.tol, self.random_state = cache_size, tol, random_state

    def _fit_binary(self, X, labels, gamma):
        probe = X[:min(16, len(X))]
        if self.backend == "cuda":
            import cupy as cp
            from cuml.svm import SVC as CuSVC
            gpu_X = cp.asarray(X, dtype=cp.float32, order="F")
            gpu_y = cp.asarray(labels, dtype=cp.float32)
            model = CuSVC(C=self.C, kernel=self.kernel, gamma=gamma,
                          tol=self.tol, cache_size=self.cache_size, output_type="cupy")
            model.fit(gpu_X, gpu_y)
            status = svm_host_array(getattr(model, "fit_status_", 0)).reshape(-1)[0]
            if status != 0:
                raise RuntimeError(f"cuML binary SVM fit failed with status {status}")
            native = model.decision_function(cp.asarray(probe, dtype=cp.float32, order="F"))
            exported = svm_export_binary(model, probe, native, self.kernel, gamma)
            cp.cuda.runtime.deviceSynchronize()
            del model, gpu_X, gpu_y, native
        else:
            model = SVC(C=self.C, kernel=self.kernel, gamma=gamma, tol=self.tol,
                        cache_size=self.cache_size, probability=False)
            model.fit(X, labels)
            exported = svm_export_binary(model, probe, model.decision_function(probe), self.kernel, gamma)
        return exported

    def fit(self, X, y):
        X, y = np.asarray(X, np.float32), np.asarray(y)
        if X.ndim != 2 or len(X) != len(y) or not np.isfinite(X).all():
            raise ValueError("Expected finite SVM feature rows and matching labels")
        if self.backend not in ("cuda", "cpu") or self.kernel not in ("linear", "rbf") or self.C <= 0:
            raise ValueError("Use cuda/cpu, linear/rbf, and a positive C")
        if not isinstance(self.calibration_folds, (int, np.integer)) or self.calibration_folds < 2:
            raise ValueError("At least two calibration folds required")
        self.classes_, counts = np.unique(y, return_counts=True)
        if len(self.classes_) < 2 or counts.min() < 2:
            raise ValueError("SVM calibration requires at least two examples per class")
        self.n_features_in_ = X.shape[1]
        if self.gamma == "scale":
            variance = float(np.var(X, dtype=np.float64))
            gamma = 1./(X.shape[1]*variance) if variance > 0 else 1.
        elif self.gamma == "auto":
            gamma = 1./X.shape[1]
        else:
            gamma = float(self.gamma)
            if gamma <= 0:
                raise ValueError("gamma must be positive")
        self.resolved_gamma_ = gamma
        self.runtime_ = cuda_svm_libraries()[2] if self.backend == "cuda" else dict(backend="cpu")
        self.binary_models_ = []
        self.calibration_assignments_ = []
        for pair_index, (first, second) in enumerate(itertools.combinations(range(len(self.classes_)), 2)):
            indices = np.flatnonzero(np.isin(y, self.classes_[[first, second]]))
            pair_X = X[indices]
            pair_y = (y[indices] == self.classes_[second]).astype(int)
            folds = min(self.calibration_folds, int(np.bincount(pair_y).min()))
            splitter = StratifiedKFold(folds, shuffle=True, random_state=self.random_state+pair_index*1009)
            scores, coverage = np.empty(len(indices)), np.zeros(len(indices), int)
            assignments = []
            for train, validation in splitter.split(pair_X, pair_y):
                parameters = self._fit_binary(pair_X[train], pair_y[train], gamma)
                scores[validation] = svm_portable_margin(parameters, pair_X[validation])
                coverage[validation] += 1
                assignments.append(dict(train=indices[train], validation=indices[validation]))
            if not np.all(coverage == 1):
                raise AssertionError("Calibration must give exactly one held-out margin per training row")
            sigmoid = svm_sigmoid_fit(scores, pair_y)
            final = self._fit_binary(pair_X, pair_y, gamma)
            self.binary_models_.append(dict(first=first, second=second, parameters=final, sigmoid=sigmoid))
            self.calibration_assignments_.append(assignments)
        self.training_backend_ = self.backend
        return self

    def predict_proba(self, X):
        if not hasattr(self, "binary_models_"):
            raise ValueError("Fit the SVM before prediction")
        X = np.asarray(X, np.float32)
        if X.ndim != 2 or X.shape[1] != self.n_features_in_ or not np.isfinite(X).all():
            raise ValueError("SVM prediction feature count or values differ from training")
        pairwise = np.zeros((len(X), len(self.classes_), len(self.classes_)))
        for member in self.binary_models_:
            scores = svm_portable_margin(member["parameters"], X)
            sigmoid = member["sigmoid"]
            positive = np.clip(expit(sigmoid["slope"]*(scores-sigmoid["mean"])/sigmoid["scale"]+sigmoid["bias"]), 1e-7, 1-1e-7)
            first, second = member["first"], member["second"]
            pairwise[:, first, second] = 1.-positive
            pairwise[:, second, first] = positive
        return svm_pairwise_coupling(pairwise)

    def predict(self, X):
        return self.classes_[self.predict_proba(X).argmax(axis=1)]

    def metadata(self):
        return dict(training_backend=self.training_backend_, inference_backend="portable_numpy_cpu",
                    multiclass="explicit_one_vs_one", binary_models=len(self.binary_models_),
                    probability_method="training_only_classifier_OOF_Platt_sigmoid_and_Wu_Lin_Weng_coupling",
                    calibration_folds_requested=self.calibration_folds, runtime=self.runtime_,
                    kernel=self.kernel, C=self.C, resolved_gamma=self.resolved_gamma_,
                    gpu_objects_saved=False, native_margin_export_checked=True)


def cuda_svm_preflight():
    """Actually fit four-class linear/RBF GPU SVMs and validate portable prediction."""
    cp, _, runtime = cuda_svm_libraries()
    rng = np.random.default_rng(42)
    labels = np.tile(np.arange(4), 12)
    features = rng.normal(0., .15, (len(labels), 8)).astype(np.float32)
    features[np.arange(len(labels)), labels] += 3.
    for kernel in ("linear", "rbf"):
        estimator = PortableCudaSVC(kernel=kernel, backend="cuda", calibration_folds=2).fit(features, labels)
        probabilities = estimator.predict_proba(features)
        if probabilities.shape != (48, 4) or not np.isfinite(probabilities).all() or not np.allclose(probabilities.sum(1), 1.):
            raise RuntimeError(f"CUDA {kernel} SVM preflight produced invalid probabilities")
        if (probabilities.argmax(1) == labels).mean() < .95:
            raise RuntimeError(f"CUDA {kernel} SVM preflight failed a simple separable-signal check")
    cp.cuda.runtime.deviceSynchronize()
    print(f"CUDA SVM CHECK PASSED: {runtime['name']} (device {runtime['device']}); linear/RBF binary fitting and four-class probabilities verified.")
    return runtime


def classical_data(views, spec):
    family = spec["representation"] if spec["representation"] in ("eight_window", "bandpower_eight_window") else "classical"
    return views.get(family, spec["view"])[:, 0]


def classical_extractor(spec):
    if spec["representation"] in ("eight_window", "bandpower_eight_window"):
        return EightWindowFeatures(
            representation=spec["representation"], n_windows=spec.get("n_windows", 8),
            wavelet=spec.get("wavelet", "db2"), wavelet_level=spec.get("wavelet_level", 3),
            ar_order=spec.get("ar_order", 4), input_sfreq=256.,
            feature_sfreq=spec.get("feature_sfreq", 128.))
    return SpatialFeatures(spec["representation"], spec.get("rank", 16), spec.get("components", 4))

## Classical model construction and controlled candidate search

Each candidate fits the appropriate extractor and training-only reduction. Cache identities include all wavelet and AR settings.

In [ ]:
def build_classical(spec, seed):
    if spec["classifier"] == "lda":
        classifier = LinearDiscriminantAnalysis(solver="lsqr", shrinkage="auto")
    elif spec.get("svm_backend", "cpu") == "cuda":
        classifier = PortableCudaSVC(
            kernel=spec["kernel"], C=spec["C"], gamma=spec.get("gamma", "scale"),
            backend="cuda", calibration_folds=spec.get("svm_calibration_folds", 3),
            cache_size=spec.get("svm_cache_size", 512.), tol=spec.get("svm_tol", 1e-3),
            random_state=seed)
    else:
        classifier = SVC(kernel=spec["kernel"], C=spec["C"], gamma=spec.get("gamma", "scale"),
                         probability=True, random_state=seed, cache_size=512)
    return Pipeline([
        ("features", classical_extractor(spec)),
        ("compact", CompactFeatures(spec["dimension"], "pca" if spec["representation"] == "riemann" else "anova")),
        ("classifier", classifier),
    ])


def candidate_list(cfg, pipeline):
    override = getattr(cfg, f"{pipeline}_candidates")
    if override is not None:
        candidates = [dict(c) for c in override]
    elif pipeline == "classical":
        representations = ("bandpower", "csp", "riemann")
        views = ("full",) if cfg.profile == "smoke" else ("full", "late")
        candidates = []
        classifiers = (("svm", "linear", 1.), ("svm", "rbf", 1.), ("lda", "linear", 1.))
        if cfg.profile == "full":
            classifiers += (("svm", "linear", .1), ("svm", "rbf", 10.))
        dimensions = (32,) if cfg.profile != "full" else (16, 64)
        for representation, view, dimension, (classifier, kernel, C) in itertools.product(representations, views, dimensions, classifiers):
            candidates.append(dict(representation=representation, view=view, dimension=dimension,
                                   rank=16, components=4, classifier=classifier, kernel=kernel, C=C, gamma="scale"))
        if cfg.profile == "smoke":
            candidates = [c for c in candidates if c["classifier"] == "svm" and c["kernel"] == "linear"]
    elif pipeline == "eegnet":
        candidates = [dict(architecture="eegnet", view="full", width=8, kernel=64, bins=4, dropout=.25, lr=.001),
                      dict(architecture="fbcnet", view="full", width=4, kernel=64, bins=4, dropout=.25, lr=.001)]
        if cfg.profile != "smoke":
            candidates += [dict(architecture="eegnet", view="late", width=8, kernel=32, bins=4, dropout=.5, lr=.001),
                           dict(architecture="fbcnet", view="late", width=4, kernel=64, bins=4, dropout=.5, lr=.001)]
        if cfg.profile == "full":
            candidates += [dict(architecture="eegnet", view="full", width=16, kernel=32, bins=8, dropout=.5, lr=.0003),
                           dict(architecture="eegnet", view="two_views", width=8, kernel=64, bins=4, dropout=.25, lr=.0003)]
    else:
        candidates = [dict(mode="probe", view="late", C=.1, dimension=32)]
        if cfg.profile != "smoke":
            candidates += [dict(mode="probe", view="two_views", C=1., dimension=32),
                           dict(mode="partial", view="late", lr=.0003, encoder_lr=.00001),
                           dict(mode="full", view="late", lr=.0003, encoder_lr=.000003)]
        if cfg.profile == "full":
            candidates += [dict(mode="probe", view="full", C=.01, dimension=64),
                           dict(mode="partial", view="two_views", lr=.0003, encoder_lr=.00001)]
    if override is None and pipeline == "classical":
        additions = []
        views = ("full",) if cfg.profile == "smoke" else ("full", "late")
        dimensions = (16, 32) if cfg.profile == "full" else (32,)
        orders = (1, 4) if cfg.profile == "full" else (4,)
        kernels = ("linear",) if cfg.profile == "smoke" else ("linear", "rbf")
        for representation, view, dimension, order, kernel in itertools.product(
                ("eight_window", "bandpower_eight_window"), views, dimensions, orders, kernels):
            additions.append(dict(representation=representation, view=view, dimension=dimension,
                                  classifier="svm", kernel=kernel, C=1., gamma="scale",
                                  n_windows=8, wavelet="db2", wavelet_level=3,
                                  ar_order=order, feature_sfreq=128.))
        if cfg.classical_feature_set == "eight_window":
            candidates = additions
        elif cfg.classical_feature_set == "combined":
            candidates += additions
    if pipeline == "classical":
        for candidate in candidates:
            if candidate.get("classifier") == "svm":
                candidate.setdefault("svm_backend", cfg.svm_backend)
                candidate.setdefault("svm_calibration_folds", cfg.svm_calibration_folds)
                candidate.setdefault("svm_cache_size", cfg.svm_cache_size)
                candidate.setdefault("svm_tol", cfg.svm_tol)
                if candidate["svm_backend"] not in ("cpu", "cuda"):
                    raise ValueError("Unknown SVM backend")
    if not candidates or len({digest(c) for c in candidates}) != len(candidates):
        raise ValueError(f"{pipeline}: candidate list must be nonempty and unique")
    for candidate in candidates:
        if candidate.get("view") not in WINDOWS:
            raise ValueError(f"Invalid window in {candidate}")
        if pipeline == "classical" and candidate["view"] == "two_views":
            raise ValueError("Classical features use one window with temporal bins; use full, late, or early")
    return candidates

## Scratch networks that retain temporal information

The EEGNet-inspired model retains four or eight ordered temporal bins. The filter-bank
challenger learns separate spatial filters per band and classifies binned log variances.
Inputs are resampled to 128 Hz after filtering. Channel scaling is estimated across training
trials and samples, preserving amplitude differences between trials; there is no per-trial
unit-variance normalization. Mild noise, amplitude scaling, and channel dropout affect
training batches only. Both architectures can train on a T4.

In [ ]:
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader


class TemporalEEGNet(nn.Module):
    """EEGNet-inspired network with an ordered temporal-bin classifier."""
    def __init__(self, n_channels, width=8, kernel=64, bins=4, dropout=.25):
        super().__init__()
        depth = 2 * width
        self.temporal = nn.Sequential(nn.Conv2d(1, width, (1, kernel), padding="same", bias=False),
                                      nn.BatchNorm2d(width))
        self.spatial_conv = nn.Conv2d(width, depth, (n_channels, 1), groups=width, bias=False)
        self.spatial = nn.Sequential(nn.BatchNorm2d(depth), nn.ELU(), nn.AvgPool2d((1, 4)), nn.Dropout(dropout))
        self.temporal_refine = nn.Sequential(
            nn.Conv2d(depth, depth, (1, 16), padding="same", groups=depth, bias=False),
            nn.Conv2d(depth, depth, 1, bias=False), nn.BatchNorm2d(depth), nn.ELU(),
            nn.AvgPool2d((1, 4)), nn.Dropout(dropout), nn.AdaptiveAvgPool2d((1, bins)))
        self.classifier = nn.Linear(depth * bins, 4)

    def forward(self, X):
        out = self.temporal(X.unsqueeze(1))
        out = self.spatial(self.spatial_conv(out))
        return self.classifier(self.temporal_refine(out).flatten(1))

    def constrain(self):
        with torch.no_grad():
            w = self.spatial_conv.weight
            norm = w.flatten(1).norm(dim=1).clamp_min(1e-8)
            w.mul_((norm.clamp(max=1.) / norm).view(-1, 1, 1, 1))


class CompactFilterBankNet(nn.Module):
    """Small FBCNet-inspired challenger; not claimed to replicate FBCNet."""
    def __init__(self, n_channels, width=4, bins=4, dropout=.25):
        super().__init__()
        self.bins = bins
        self.spatial_conv = nn.Conv2d(len(BANDS), len(BANDS)*width, (n_channels, 1), groups=len(BANDS), bias=False)
        self.norm = nn.BatchNorm2d(len(BANDS)*width)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(len(BANDS)*width*bins, 4)

    def forward(self, X):
        X = torch.nn.functional.elu(self.norm(self.spatial_conv(X))).squeeze(2)
        temporal = [chunk.float().var(dim=-1, unbiased=False).clamp_min(1e-6).log()
                    for chunk in torch.tensor_split(X, self.bins, dim=-1)]
        return self.classifier(self.dropout(torch.cat(temporal, dim=1)))

    constrain = TemporalEEGNet.constrain


def scratch_model(spec, n_channels):
    common = dict(n_channels=n_channels, width=spec["width"], bins=spec["bins"], dropout=spec["dropout"])
    if spec["architecture"] == "eegnet":
        return TemporalEEGNet(kernel=spec["kernel"], **common)
    if spec["architecture"] == "fbcnet":
        return CompactFilterBankNet(**common)
    raise ValueError(spec["architecture"])


def choose_device(cfg):
    device = "cuda" if cfg.device == "auto" and torch.cuda.is_available() else ("cpu" if cfg.device == "auto" else cfg.device)
    if device.startswith("cuda") and not torch.cuda.is_available():
        raise RuntimeError("CUDA requested but unavailable. Enable a Kaggle GPU or use device='cpu'.")
    return torch.device(device)


def fit_normalizer(data):
    # Axes: trial, view, [band], electrode, sample. Preserve differences between trials.
    axes = (0, 1, data.ndim-1)
    return dict(mean=data.mean(axis=axes, keepdims=True, dtype=np.float64).astype(np.float32),
                std=np.maximum(data.std(axis=axes, keepdims=True, dtype=np.float64), 1e-6).astype(np.float32))


def normalize(data, norm):
    return np.ascontiguousarray(data if norm is None else (data - norm["mean"]) / norm["std"], np.float32)


def augment_batch(X):
    shape = (len(X),) + (1,) * (X.ndim-1)
    X = X * torch.empty(shape, device=X.device).uniform_(.95, 1.05)
    scale = X.std(dim=-1, keepdim=True).clamp_min(1e-6)
    X = X + torch.randn_like(X) * scale * .01
    # Same channel mask across views/bands, one mask per original trial.
    mask_shape = (len(X),) + (1,) * (X.ndim-3) + (X.shape[-2], 1)
    return X * (torch.rand(mask_shape, device=X.device) > .03)


def trial_logits(model, batch):
    n, views = batch.shape[:2]
    logits = model(batch.flatten(0, 1)).float()
    if logits.shape != (n*views, 4):
        raise ValueError(f"Expected four class logits, got {tuple(logits.shape)}")
    # One loss and one prediction per original trial, even when it has two views.
    return logits.reshape(n, views, 4).mean(dim=1)


def predict_neural(model, X, batch_size, device):
    model.to(device).eval()
    probabilities = []
    with torch.inference_mode():
        for start in range(0, len(X), batch_size):
            batch = torch.from_numpy(np.ascontiguousarray(X[start:start+batch_size])).to(device)
            probabilities.append(trial_logits(model, batch).softmax(dim=-1).cpu().numpy())
    probability = np.concatenate(probabilities).astype(np.float64)
    return probability / probability.sum(1, keepdims=True)

## Pretrained LaBraM with checked weights and geometry-only montage adaptation

LaBraM is pinned to an immutable Hugging Face checkpoint revision, and its downloaded file
hash is recorded. Loading fails if any learned encoder tensor is missing or incompatible;
there is no silent random-initialization fallback.

BioSemi labels such as `A1/A2` are not canonical mastoid labels. Interpolation uses electrode
geometry with `interpolation_mode="always"`; its matrix stays frozen. The adapter maps to
128 canonical positions, but interpolation cannot create measurements that were not recorded.
The library's canonical set also contains approximated bipolar positions, so this remains an
experimental montage adaptation, not an exact reconstruction of those recording derivations.

Input is resampled to 200 Hz and scaled as microvolts/100. Two-second views contain exactly
two 200-sample patches. The full 2.5-second view receives 0.5 seconds of **right padding**.
Short inputs retain the first pretrained temporal embeddings rather than stretching the
full pretraining duration across a short trial. All adaptations are saved in the load report.

Smoke uses a frozen probe. Pilot/full compare frozen, partial, and full adaptation as
separate candidates; a later stage cannot overwrite a better earlier strategy automatically.

In [ ]:
def labram_channels(names):
    import mne
    info = mne.create_info(names, sfreq=200., ch_types="eeg")
    info.set_montage(mne.channels.make_standard_montage("biosemi128"))
    return info["chs"]


def instantiate_labram(names, n_times):
    try:
        from braindecode.models import InterpolatedLaBraM
    except ImportError as exc:
        raise RuntimeError("Install braindecode[hub]>=1.8,<2 and restart the kernel to use LaBraM") from exc
    # Important: BioSemi A1/A2 are not the canonical mastoid A1/A2.
    # Always interpolate geometrically; never identify electrodes by those names.
    return InterpolatedLaBraM(chs_info=labram_channels(names), n_times=n_times, n_outputs=4,
                              sfreq=200., patch_size=200, interpolation_mode="always", trainable=False)


def resolve_labram(cfg):
    from huggingface_hub import HfApi, hf_hub_download
    directory = Path(cfg.output_root) / "pretrained"
    lock = directory / f"labram_{digest([cfg.labram_repo, cfg.labram_revision])}.json"
    if lock.exists():
        metadata = json.loads(lock.read_text())
        revision = metadata["revision"]
    elif re.fullmatch(r"[0-9a-f]{40}", cfg.labram_revision):
        revision = cfg.labram_revision
    elif cfg.labram_offline:
        raise RuntimeError("Offline LaBraM needs a saved pretrained lock file or an explicit immutable commit SHA")
    else:
        revision = HfApi().model_info(cfg.labram_repo, revision=cfg.labram_revision).sha
    path = hf_hub_download(cfg.labram_repo, "model.safetensors", revision=revision,
                           cache_dir=str(directory / "hub"), local_files_only=cfg.labram_offline)
    metadata = dict(repo=cfg.labram_repo, revision=revision, weights_sha256=file_sha(path))
    if lock.exists() and json.loads(lock.read_text()) != metadata:
        raise RuntimeError("Pretrained checkpoint hash changed")
    save_json(metadata, lock)
    return str(path), metadata


def adapted_labram(names, n_times, pretrained_path):
    from safetensors.torch import load_file
    model = instantiate_labram(names, n_times)
    incoming = load_file(pretrained_path, device="cpu")
    target = model.state_dict()
    adapted, missing, resized = {}, [], []
    for name, value in target.items():
        if name.startswith(("final_layer.", "interpolation_layer.")):
            continue
        if name not in incoming:
            # Deterministic metadata buffers can be rebuilt; learned weights cannot.
            if name in dict(model.named_parameters()):
                missing.append(name)
            continue
        old = incoming[name]
        if old.shape == value.shape:
            adapted[name] = old
        elif name == "temporal_embedding" and old.shape[0] == value.shape[0] and old.shape[2] == value.shape[2] and old.shape[1] >= value.shape[1]:
            # Keep the pretrained first positions; do not stretch 15-second time over 2 seconds.
            adapted[name] = old[:, :value.shape[1]].clone()
            resized.append(dict(name=name, source=list(old.shape), target=list(value.shape), method="prefix_slice"))
        else:
            missing.append(f"{name}: {tuple(old.shape)} -> {tuple(value.shape)}")
    if missing:
        raise RuntimeError(f"Pretrained encoder mismatch (no random fallback): {missing}")
    parameter_names = set(dict(model.named_parameters()))
    expected = {name for name in parameter_names if not name.startswith(("final_layer.", "interpolation_layer."))}
    if not expected <= set(adapted):
        raise RuntimeError("Not every encoder parameter was loaded")
    model.load_state_dict(adapted, strict=False)
    matrix = model.interpolation_layer.matrix.detach().cpu().numpy()
    if matrix.shape != (128, len(names)) or not np.isfinite(matrix).all():
        raise ValueError("Invalid LaBraM channel interpolation matrix")
    report = dict(encoder_parameters_loaded=len(expected), tensors_loaded=len(adapted),
                  temporal_adaptation=resized, projection_shape=list(matrix.shape),
                  projection_sha256=hashlib.sha256(matrix.tobytes()).hexdigest(), interpolation_mode="always")
    return model, report


def configure_labram(model, mode):
    for parameter in model.parameters():
        parameter.requires_grad_(mode == "full")
    if mode in ("head", "partial"):
        modules = [model.final_layer]
        if mode == "partial":
            modules += [*model.blocks[-2:], model.norm]
            if getattr(model, "fc_norm", None) is not None:
                modules.append(model.fc_norm)
        for module in modules:
            for parameter in module.parameters():
                parameter.requires_grad_(True)
    for parameter in model.interpolation_layer.parameters():
        parameter.requires_grad_(False)


def extract_labram_features(model, data, cfg):
    # Deterministic frozen encoder, no fit on this subject (even unlabeled).
    device = choose_device(cfg)
    model.final_layer = nn.Identity()
    model.to(device).eval()
    outputs = []
    with torch.inference_mode():
        for start in range(0, len(data), cfg.labram_batch_size):
            X = torch.from_numpy(np.ascontiguousarray(data[start:start+cfg.labram_batch_size])).to(device)
            features = model(X.flatten(0, 1)).float()
            outputs.append(features.reshape(len(X), -1).cpu().numpy())
    features = np.concatenate(outputs)
    if not np.isfinite(features).all():
        raise ValueError("Nonfinite pretrained embeddings")
    return features


def get_frozen_features(views, spec, names, cfg, data_identity, pretrained):
    data = views.get("labram", spec["view"])
    path, metadata = pretrained
    feature_identity = {key: value for key, value in data_identity.items() if key != "labels"}
    key = digest(dict(data=feature_identity, channels=names, view=spec["view"], pretrained=metadata,
                      code=code_identity(), versions=package_versions()))
    directory = Path(cfg.output_root) / "frozen_features" / key
    feature_path = directory / "features.npy"
    model_path = directory / "encoder.pt"
    if cfg.resume and feature_path.exists() and model_path.exists() and (directory / "complete.json").exists():
        return np.load(feature_path), model_path
    model, report = adapted_labram(names, data.shape[-1], path)
    # Save before replacing the head with Identity; restore logic expects the four-class skeleton.
    directory.mkdir(parents=True, exist_ok=True)
    torch_save(dict(state={k: v.detach().cpu() for k, v in model.state_dict().items()},
                    names=names, n_times=data.shape[-1], pretrained=metadata, report=report), model_path)
    features = extract_labram_features(model, data, cfg)
    with feature_path.with_suffix(".tmp").open("wb") as f:
        np.save(f, features)
    feature_path.with_suffix(".tmp").replace(feature_path)
    save_json(dict(identity=key, report=report, shape=list(features.shape)), directory / "complete.json")
    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return features, model_path


def build_probe(spec):
    return Pipeline([("compact", CompactFeatures(spec["dimension"], "pca")),
                     ("classifier", LogisticRegression(C=spec["C"], max_iter=3000))])

## Neural training, inner early stopping, and outer-training refits

Maximum epochs are a ceiling. Search training stops after validation loss fails to improve
for the patience setting, subject to the minimum search duration. The selected checkpoint
can be earlier than that minimum. Final refits train for the chosen duration without seeing
outer test loss or labels. This is why a completed model need not run to the maximum epochs.

The `full` profile averages three independent neural refits. LaBraM frozen probes remain
deterministic classical fits and do not need three identical repeats. CUDA uses mixed precision
when enabled. Set `cfg.amp=False` if your hardware produces nonfinite losses. Seeds and
deterministic settings improve repeatability; hardware/library changes need not be bit-identical.

In [ ]:
def torch_save(value, path):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_suffix(path.suffix + ".tmp")
    torch.save(value, temp); temp.replace(path)


def cpu_state(model):
    return {key: tensor.detach().cpu().clone() for key, tensor in model.state_dict().items()}


def neural_fit(model, X, y, spec, cfg, seed, pipeline, validation=None, epochs=None):
    seed_everything(seed)
    device = choose_device(cfg)
    model.to(device)
    if pipeline == "labram":
        configure_labram(model, spec["mode"])
        head, encoder = [], []
        for name, p in model.named_parameters():
            if p.requires_grad:
                (head if name.startswith("final_layer.") else encoder).append(p)
        groups = [{"params": head, "lr": spec["lr"]}]
        if encoder:
            groups.append({"params": encoder, "lr": spec["encoder_lr"]})
        optimizer = torch.optim.AdamW(groups, weight_decay=.01)
        batch_size = cfg.labram_batch_size
    else:
        optimizer = torch.optim.AdamW(model.parameters(), lr=spec["lr"], weight_decay=1e-4)
        batch_size = cfg.scratch_batch_size
    total_epochs = int(epochs or cfg.max_epochs)
    # Warmup/cosine uses the same original horizon in tuning and fixed-epoch refits.
    # Otherwise refitting for fewer epochs would change the optimization schedule.
    def schedule(epoch):
        warmup = min(5, max(1, cfg.max_epochs // 10))
        if epoch < warmup:
            return (epoch+1) / warmup
        return .05 + .95 * .5 * (1 + math.cos(math.pi * min(1., (epoch-warmup)/max(1, cfg.max_epochs-warmup))))
    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, schedule)
    dataset = TensorDataset(torch.from_numpy(np.ascontiguousarray(X)), torch.as_tensor(y, dtype=torch.long))
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True, drop_last=False,
                        generator=torch.Generator().manual_seed(seed), num_workers=0, pin_memory=device.type == "cuda")
    use_amp = cfg.amp and device.type == "cuda"
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    best_loss, best_epoch, best_state, stale = np.inf, 1, None, 0
    history = []
    progress = tqdm(range(1, total_epochs+1), desc=f"{pipeline} {'inner epochs' if validation is not None else 'refit epochs'}",
                    leave=False, mininterval=2.)
    for epoch in progress:
        model.train()
        if pipeline == "labram" and spec["mode"] != "full":
            model.eval()  # keep frozen blocks deterministic
            model.final_layer.train()
            if spec["mode"] == "partial":
                for block in model.blocks[-2:]:
                    block.train()
        loss_sum, correct, count = 0., 0, 0
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            if pipeline == "eegnet":
                xb = augment_batch(xb)
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
                logits = trial_logits(model, xb)
                loss = nn.functional.cross_entropy(logits, yb)
            if not torch.isfinite(loss):
                raise FloatingPointError("Nonfinite training loss; inspect scaling or disable AMP")
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.)
            scaler.step(optimizer); scaler.update()
            if hasattr(model, "constrain"):
                model.constrain()
            loss_sum += loss.item() * len(yb)
            correct += int((logits.argmax(1) == yb).sum())
            count += len(yb)
        row = dict(epoch=epoch, train_loss=loss_sum/count, train_accuracy=correct/count,
                   learning_rate=max(g["lr"] for g in optimizer.param_groups))
        scheduler.step()
        if validation is not None:
            Xv, yv = validation
            prob = predict_neural(model, Xv, batch_size, device)
            score = metrics(yv, prob.argmax(1), prob)
            row.update({"val_"+k: v for k, v in score.items()})
            if score["log_loss"] < best_loss - 1e-5:
                best_loss, best_epoch, best_state, stale = score["log_loss"], epoch, cpu_state(model), 0
            else:
                stale += 1
        history.append(row)
        status = dict(train_loss=f"{row['train_loss']:.3f}")
        if validation is not None:
            status.update(val_loss=f"{row['val_log_loss']:.3f}", best_epoch=best_epoch)
        progress.set_postfix(status, refresh=False)
        if validation is not None and epoch >= cfg.min_epochs and stale >= cfg.patience:
            break
    if validation is not None:
        if best_state is None:
            raise RuntimeError("No finite validation checkpoint")
        model.load_state_dict(best_state)
    else:
        best_epoch = total_epochs
    return model, int(best_epoch), pd.DataFrame(history)


def neural_data(views, spec, pipeline):
    family = spec["architecture"] if pipeline == "eegnet" else "labram"
    return views.get(family, spec["view"])


def new_neural_model(spec, pipeline, names, n_times, pretrained, seed):
    seed_everything(seed)
    if pipeline == "eegnet":
        return scratch_model(spec, len(names)), {}
    return adapted_labram(names, n_times, pretrained[0])

## Nested model selection and portable model bundles

Each completed candidate search and outer model is saved. Resume works at candidate/outer-fold
boundaries, with additional reuse of completed neural ensemble members. An interrupted epoch
or inner candidate restarts; optimizer state is not resumed in the middle of an epoch.

Cache identities include data hashes, labels, exact splits, code, package versions, model
configuration, electrode order, and pretrained identity. Changed settings are separate
experiments. Loading trusted pickle checkpoints executes Python object deserialization.

In [ ]:
def code_identity():
    if SOURCE_SHA256 != "development":
        return SOURCE_SHA256
    return file_sha(__file__)


def fit_candidate_inner(spec, pipeline, views, y, names, plan, cfg, seed, directory,
                        pretrained, data_identity, feature_cache):
    """Receives the split plan but only fits/scores on its INNER train/validation rows."""
    inner_predictions, chosen_epochs = [], []
    if pipeline == "labram" and spec["mode"] == "probe":
        features, _ = get_frozen_features(views, spec, names, cfg, data_identity, pretrained)
    for inner_fold, split in enumerate(plan["inner"], 1):
        train, validation = np.array(split["train"]), np.array(split["validation"])
        fold_seed = int(seed + inner_fold * 7919)
        if pipeline == "classical":
            data = classical_data(views, spec)
            key = digest(dict(extraction={key: value for key, value in spec.items()
                                          if key not in ("classifier", "kernel", "C", "gamma", "dimension",
                                                         "svm_backend", "svm_calibration_folds", "svm_cache_size", "svm_tol")},
                              train=train, validation=validation))
            if key not in feature_cache:
                transform = classical_extractor(spec)
                training_features = transform.fit_transform(data[train], y[train])
                validation_features = transform.transform(data[validation])
                feature_cache[key] = (training_features, validation_features)
            training_features, validation_features = feature_cache[key]
            pipeline_model = build_classical(spec, fold_seed)
            # Features above are already fitted separately for this exact training split.
            classifier = Pipeline(pipeline_model.steps[1:]).fit(training_features, y[train])
            probability = classifier.predict_proba(validation_features)
        elif pipeline == "labram" and spec["mode"] == "probe":
            classifier = build_probe(spec).fit(features[train], y[train])
            probability = classifier.predict_proba(features[validation])
        else:
            data = neural_data(views, spec, pipeline)
            norm = fit_normalizer(data[train]) if pipeline == "eegnet" else None
            model, _ = new_neural_model(spec, pipeline, names, data.shape[-1], pretrained, fold_seed)
            model, duration, history = neural_fit(model, normalize(data[train], norm), y[train], spec, cfg,
                                                   fold_seed, pipeline, validation=(normalize(data[validation], norm), y[validation]))
            probability = predict_neural(model, normalize(data[validation], norm),
                                         cfg.labram_batch_size if pipeline == "labram" else cfg.scratch_batch_size,
                                         choose_device(cfg))
            chosen_epochs.append(duration)
            save_csv(history, directory / f"inner_{inner_fold:02d}_history.csv")
            del model
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
        rows = pd.DataFrame(dict(trial_index=validation, inner_fold=inner_fold, y_true=y[validation], y_pred=probability.argmax(1)))
        for c in CLASSES:
            rows[f"p_{c}"] = probability[:, c]
        inner_predictions.append(rows)
    oof = pd.concat(inner_predictions).sort_values("trial_index")
    if oof.trial_index.duplicated().any() or set(oof.trial_index) != set(plan["train"]):
        raise AssertionError("Inner OOF did not cover exactly the outer training set")
    probability = oof[[f"p_{c}" for c in CLASSES]].to_numpy()
    score = metrics(oof.y_true, probability.argmax(1), probability)
    result = dict(spec=spec, score=score, epochs=chosen_epochs,
                  refit_epochs=max(1, int(np.rint(np.median(chosen_epochs)))) if chosen_epochs else None)
    save_csv(oof, directory / "inner_oof.csv")
    save_json(result, directory / "complete.json")
    return result


def select_candidate(specs, pipeline, views, y, names, plan, cfg, seed, directory, pretrained, data_identity):
    results, feature_cache = [], {}
    progress = tqdm(specs, desc=f"{pipeline} inner search", leave=False)
    for spec in progress:
        candidate_directory = directory / "inner_search" / digest(spec)
        candidate_directory.mkdir(parents=True, exist_ok=True)
        completed = candidate_directory / "complete.json"
        if cfg.resume and completed.exists():
            result = json.loads(completed.read_text())
        else:
            result = fit_candidate_inner(spec, pipeline, views, y, names, plan, cfg, seed,
                                          candidate_directory, pretrained, data_identity, feature_cache)
        results.append(result)
        progress.set_postfix(bal=f"{result['score']['balanced_accuracy']:.3f}")
    # Use all inner OOF trials, rather than unweighted means of unequal fold scores.
    best = sorted(results, key=lambda r: (-r["score"]["balanced_accuracy"], r["score"]["log_loss"], digest(r["spec"])))[0]
    leaderboard = pd.DataFrame([dict(candidate=digest(r["spec"]), **r["score"],
                                      refit_epochs=r["refit_epochs"], specification=json.dumps(r["spec"], sort_keys=True)) for r in results])
    save_csv(leaderboard, directory / "inner_leaderboard.csv")
    save_json(best, directory / "selected.json")
    return best


def refit_and_predict(selected, pipeline, views, y, names, plan, cfg, seed, directory, pretrained, data_identity):
    spec = selected["spec"]
    train, test = np.asarray(plan["train"]), np.asarray(plan["test"])
    bundle = dict(version=VERSION, code=code_identity(), pipeline=pipeline, spec=spec,
                  channel_names=names, input_sfreq=cfg.expected_sfreq, action_interval=[1., 3.5],
                  input_units="microvolts", train_indices=train, test_indices=test,
                  selected_inner_score=selected["score"], package_versions=package_versions())
    if pipeline == "classical":
        data = classical_data(views, spec)
        model = build_classical(spec, seed).fit(data[train], y[train])
        probability = model.predict_proba(data[test])
        train_probability = model.predict_proba(data[train])
        bundle["estimator"] = model
        if isinstance(model.named_steps["classifier"], PortableCudaSVC):
            backend_metadata = model.named_steps["classifier"].metadata()
            bundle["svm_backend_metadata"] = backend_metadata
            save_json(backend_metadata, directory / "svm_backend.json")
        reducer = model.named_steps["compact"]
        np.savez(directory / "feature_reduction.npz", kept_nonconstant=reducer.variance_.get_support(indices=True),
                 selected_after_variance=reducer.reducer_.get_support(indices=True) if hasattr(reducer.reducer_, "get_support") else np.array([], int),
                 pca_components=getattr(reducer.reducer_, "components_", np.empty((0, 0))))
        if isinstance(model.named_steps["features"], EightWindowFeatures):
            extractor = model.named_steps["features"]
            schema = extractor.describe(names)
            start_time = WINDOWS[spec["view"]][0][0]
            schema["start_time_s"] = start_time + schema.start_offset_s
            schema["stop_time_s"] = start_time + schema.stop_offset_s
            save_csv(schema, directory / "raw_feature_schema.csv")
            kept = reducer.variance_.get_support(indices=True)
            selected_columns = kept[reducer.reducer_.get_support(indices=True)]
            selected_schema = schema.iloc[selected_columns].copy()
            selected_schema["training_ANOVA_F"] = reducer.reducer_.scores_[reducer.reducer_.get_support(indices=True)]
            save_csv(selected_schema, directory / "selected_feature_schema.csv")
            feature_metadata = extractor.metadata() | dict(
                view=spec["view"], view_interval=WINDOWS[spec["view"]][0],
                final_feature_count=len(selected_columns), reduction="training_only_variance_scaling_ANOVA",
                feature_order="window_then_electrode_then_AR_entropy_log_variance_then_optional_bandpower")
            save_json(feature_metadata, directory / "feature_extraction.json")
            bundle["feature_extraction"] = feature_metadata
    elif pipeline == "labram" and spec["mode"] == "probe":
        features, encoder_path = get_frozen_features(views, spec, names, cfg, data_identity, pretrained)
        model = build_probe(spec).fit(features[train], y[train])
        probability, train_probability = model.predict_proba(features[test]), model.predict_proba(features[train])
        bundle.update(estimator=model, encoder_relative_path=relative_path(encoder_path, directory), pretrained=pretrained[1])
    else:
        data = neural_data(views, spec, pipeline)
        norm = fit_normalizer(data[train]) if pipeline == "eegnet" else None
        Xtrain, Xtest = normalize(data[train], norm), normalize(data[test], norm)
        predictions, training_predictions, members = [], [], []
        for member in cfg.ensemble_seeds:
            member_seed = int(seed + int(member)*104729)
            model_path = directory / f"member_{member}.pt"
            history_path = directory / f"member_{member}_refit_history.csv"
            if cfg.resume and model_path.exists() and history_path.exists():
                payload = torch.load(model_path, map_location="cpu", weights_only=False)
                model = scratch_model(spec, len(names)) if pipeline == "eegnet" else instantiate_labram(names, data.shape[-1])
                model.load_state_dict(payload["state"], strict=True)
            else:
                model, load_report = new_neural_model(spec, pipeline, names, data.shape[-1], pretrained, member_seed)
                model, _, history = neural_fit(model, Xtrain, y[train], spec, cfg, member_seed, pipeline,
                                                epochs=selected["refit_epochs"])
                payload = dict(state=cpu_state(model), norm=norm, seed=member_seed, spec=spec,
                               n_times=data.shape[-1], report=load_report,
                               pretrained=None if pretrained is None else pretrained[1])
                torch_save(payload, model_path)
                save_csv(history, history_path)
            batch_size = cfg.labram_batch_size if pipeline == "labram" else cfg.scratch_batch_size
            predictions.append(predict_neural(model, Xtest, batch_size, choose_device(cfg)))
            training_predictions.append(predict_neural(model, Xtrain, batch_size, choose_device(cfg)))
            members.append(model_path.name)
            del model
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
        probability = np.mean(predictions, axis=0)
        train_probability = np.mean(training_predictions, axis=0)
        bundle.update(members=members, norm=norm, n_times=data.shape[-1], refit_epochs=selected["refit_epochs"])
    save_pickle(bundle, directory / "bundle.pkl")
    return probability, metrics(y[train], train_probability.argmax(1), train_probability)


def predict_saved_fold(bundle_path, action_eeg, channel_names, sfreq, cfg=None):
    """Restore a fold model without fitting. Input is [trials,channels,time], microvolts,
    exactly the [1.0,3.5) action interval, in the explicitly provided channel order.
    Keep the complete output root so relative pretrained/ensemble paths resolve.
    """
    bundle_path = Path(bundle_path)
    bundle = read_pickle(bundle_path)
    if list(channel_names) != bundle["channel_names"] or not np.isclose(sfreq, bundle["input_sfreq"]):
        raise ValueError("Checkpoint channel order or sampling frequency does not match input")
    X = np.asarray(action_eeg, np.float32)
    if X.ndim != 3 or X.shape[1:] != (len(channel_names), round(2.5*sfreq)) or not np.isfinite(X).all():
        raise ValueError("Expected finite action-only microvolt EEG with the saved shape")
    cfg = cfg or make_config("smoke")
    views = SubjectViews(X, sfreq)
    spec, pipeline = bundle["spec"], bundle["pipeline"]
    if pipeline == "classical":
        return bundle["estimator"].predict_proba(classical_data(views, spec))
    data = neural_data(views, spec, pipeline)
    if pipeline == "labram" and spec["mode"] == "probe":
        encoder = torch.load(bundle_path.parent / bundle["encoder_relative_path"], map_location="cpu", weights_only=False)
        model = instantiate_labram(channel_names, encoder["n_times"])
        model.load_state_dict(encoder["state"], strict=True)
        return bundle["estimator"].predict_proba(extract_labram_features(model, data, cfg))
    predictions = []
    for filename in bundle["members"]:
        payload = torch.load(bundle_path.parent / filename, map_location="cpu", weights_only=False)
        model = scratch_model(spec, len(channel_names)) if pipeline == "eegnet" else instantiate_labram(channel_names, payload["n_times"])
        model.load_state_dict(payload["state"], strict=True)
        predictions.append(predict_neural(model, normalize(data, payload["norm"]),
                                          cfg.labram_batch_size if pipeline == "labram" else cfg.scratch_batch_size,
                                          choose_device(cfg)))
        del model
    return np.mean(predictions, axis=0)

## Exact OOF reports, saved error analysis, and the easy runner

Reports include accuracy, balanced accuracy, macro-F1, log loss, trial probabilities, per-class
metrics, confusion matrices, inner-search results, training histories, and resubstitution
training metrics for diagnosing training/test gaps. SVM probabilities use its internal
training-only calibration; they should not be assumed perfectly calibrated on unseen trials.

Checkpoints contain feature transforms or neural weights and normalization, plus portable
relative paths. Copy the whole output root when transferring runs to another machine.
The notebook does not automatically choose the best of the three outer-scored procedures
and present that maximum as an unbiased combined-model result.

In [ ]:
def aggregate_oof(frame):
    keys = ["pipeline", "n_channels", "subject", "cv_seed", "diagnostic"]
    rows = []
    for values, group in frame.groupby(keys, sort=True):
        if group.trial_id.duplicated().any():
            raise AssertionError(f"Duplicate OOF trials in {values}")
        probability = group[[f"p_{c}" for c in CLASSES]].to_numpy()
        rows.append(dict(zip(keys, values)) | dict(n_trials=len(group), **metrics(group.y_true, group.y_pred, probability)))
    by_seed = pd.DataFrame(rows)
    scores = ["accuracy", "balanced_accuracy", "macro_f1", "log_loss"]
    # A subject, not a fold or a repeated seed, is the independent summary unit.
    by_subject = by_seed.groupby(["pipeline", "n_channels", "subject", "diagnostic"], as_index=False)[scores].mean()
    summaries = []
    for (pipeline, channels, diagnostic), group in by_subject.groupby(["pipeline", "n_channels", "diagnostic"]):
        row = dict(pipeline=pipeline, n_channels=channels, diagnostic=diagnostic, n_subjects=len(group))
        for score in scores:
            values = group[score].to_numpy()
            row[score+"_mean"] = float(values.mean())
            row[score+"_sd"] = float(values.std(ddof=1)) if len(values) > 1 else np.nan
        if len(group) > 1:
            rng = np.random.default_rng(826)
            draws = rng.choice(group.accuracy.to_numpy(), size=(5000, len(group)), replace=True).mean(axis=1)
            row["accuracy_ci_low"], row["accuracy_ci_high"] = np.quantile(draws, [.025, .975])
        else:
            row["accuracy_ci_low"] = row["accuracy_ci_high"] = np.nan
        summaries.append(row)
    return by_seed, by_subject, pd.DataFrame(summaries)


def write_reports(frame, directory, fold_rows):
    directory = Path(directory)
    by_seed, by_subject, summary = aggregate_oof(frame)
    save_csv(frame, directory / "oof_predictions.csv")
    save_csv(pd.DataFrame(fold_rows), directory / "fold_metrics.csv")
    save_csv(by_seed, directory / "subject_seed_metrics.csv")
    save_csv(by_subject, directory / "subject_metrics.csv")
    save_csv(summary, directory / "summary.csv")
    for (pipeline, channels, subject, seed), group in frame.groupby(["pipeline", "n_channels", "subject", "cv_seed"]):
        stem = f"{pipeline}_{channels}ch_sub-{subject:02d}_seed-{seed}"
        matrix = confusion_matrix(group.y_true, group.y_pred, labels=CLASSES)
        save_csv(pd.DataFrame(matrix, columns=CLASS_NAMES).assign(true_class=CLASS_NAMES), directory / "confusion_matrices" / f"{stem}.csv")
        report = classification_report(group.y_true, group.y_pred, labels=CLASSES, target_names=CLASS_NAMES,
                                       output_dict=True, zero_division=0)
        save_json(report, directory / "classification_reports" / f"{stem}.json")
    return dict(oof=frame, subject_seed_metrics=by_seed, subject_metrics=by_subject, summary=summary, directory=directory)


def preview_run(cfg):
    cfg.validate()
    available = sorted({r[0] for r in discover_recordings(cfg.data_root)})
    subjects = available if cfg.subject_ids is None else list(cfg.subject_ids)
    missing = set(subjects) - set(available)
    if missing:
        raise ValueError(f"Requested subjects are missing: {sorted(missing)}")
    outer_tasks = len(subjects)*len(cfg.channel_counts)*len(cfg.cv_seeds)*cfg.outer_folds
    rows = []
    for pipeline in cfg.pipelines:
        specs = candidate_list(cfg, pipeline)
        rows.append(dict(pipeline=pipeline, candidates=len(specs), outer_tasks=outer_tasks,
                         inner_fits=outer_tasks*cfg.inner_folds*len(specs),
                         final_fits_max=outer_tasks*(1 if pipeline == "classical" else len(cfg.ensemble_seeds))))
    print(f"Profile={cfg.profile}; subjects={subjects}; channels={cfg.channel_counts}; device={choose_device(cfg)}")
    print("SMOKE IS AN EXECUTION CHECK, NOT A PERFORMANCE ESTIMATE." if cfg.profile == "smoke"
          else "Each pipeline is selected inside outer training; each original trial receives one OOF prediction per CV seed.")
    print(f"Classical features/scaling: CPU. SVM requested backend: {cfg.svm_backend}. "
          "LDA remains CPU; neural training uses CUDA when available.")
    if cfg.svm_backend == "cuda":
        print("CUDA SVM is mandatory: the runner checks actual cuML fitting; no automatic CPU fallback.")
    return pd.DataFrame(rows)


def scientific_identity(cfg, pipeline):
    config = asdict(cfg)
    for key in ("data_root", "output_root", "subject_ids", "channel_counts", "pipelines", "resume", "labram_offline"):
        config.pop(key)
    for name in ("classical", "eegnet", "labram"):
        config.pop(f"{name}_candidates")
    config["candidates"] = candidate_list(cfg, pipeline)
    return dict(config=config, code=code_identity(), versions=package_versions())


def bundle_inventory(directory):
    bundle = read_pickle(directory / "bundle.pkl")
    files = [directory / "bundle.pkl", directory / "test_predictions.csv"] + [directory / name for name in bundle.get("members", [])]
    if "encoder_relative_path" in bundle:
        files.append(directory / bundle["encoder_relative_path"])
    if "svm_backend_metadata" in bundle:
        files.append(directory / "svm_backend.json")
    if "feature_extraction" in bundle:
        files += [directory / name for name in
                  ("raw_feature_schema.csv", "selected_feature_schema.csv", "feature_extraction.json", "feature_reduction.npz")]
    return {relative_path(path, directory): file_sha(path) for path in files}


def checkpoint_complete(directory, identity):
    marker = directory / "complete.json"
    if not marker.exists() or not (directory / "test_predictions.csv").exists():
        return False
    metadata = json.loads(marker.read_text())
    if metadata.get("identity") != identity:
        return False
    inventory = metadata.get("files", {})
    return bool(inventory) and all((directory / name).exists() and file_sha(directory / name) == checksum
                                    for name, checksum in inventory.items())


def run_experiments(cfg):
    cfg.validate()
    needs_cuda_svm = "classical" in cfg.pipelines and any(
        candidate.get("classifier") == "svm" and candidate.get("svm_backend") == "cuda"
        for candidate in candidate_list(cfg, "classical"))
    svm_runtime = cuda_svm_preflight() if needs_cuda_svm else None
    print(preview_run(cfg).to_string(index=False))
    torch.set_num_threads(cfg.cpu_threads)
    available = sorted({r[0] for r in discover_recordings(cfg.data_root)})
    subjects = available if cfg.subject_ids is None else list(cfg.subject_ids)
    pretrained = resolve_labram(cfg) if "labram" in cfg.pipelines else None
    root = Path(cfg.output_root)
    source_manifest = [dict(subject=s, session=session, eeg_sha256=file_sha(eeg), events_sha256=file_sha(events))
                       for s, session, eeg, events in discover_recordings(cfg.data_root) if s in subjects]
    run_id = digest(dict(config=asdict(cfg), code=code_identity(), versions=package_versions(),
                         sources=source_manifest, pretrained=None if pretrained is None else pretrained[1]))
    directory = root / "runs" / run_id
    save_json(dict(config=asdict(cfg), versions=package_versions(), code=code_identity(),
                   sources=source_manifest, pretrained=None if pretrained is None else pretrained[1],
                   svm_cuda_preflight=svm_runtime), directory / "run_config.json")
    frames, fold_rows = [], []
    with threadpool_limits(limits=cfg.cpu_threads):
        for subject in subjects:
            X, names, meta, audit = load_subject(cfg, subject)
            if cfg.label_permutation_seed is not None:
                rng = np.random.default_rng(cfg.label_permutation_seed + int(subject)*101)
                meta["original_label"] = meta.y_true.copy()
                for session in meta.session.unique():
                    indices = np.flatnonzero(meta.session.to_numpy() == session)
                    meta.loc[indices, "y_true"] = rng.permutation(meta.loc[indices, "y_true"].to_numpy())
            data_identity = dict(audit=audit, trial_ids=meta.trial_id.tolist(), labels=meta.y_true.tolist())
            save_json(audit, directory / "data_audits" / f"subject_{subject:02d}.json")
            save_csv(meta, directory / "data_audits" / f"subject_{subject:02d}_trials.csv")
            subsets = channel_subsets(names)
            save_json({n: [names[i] for i in indices] for n, indices in subsets.items()}, directory / "channel_subsets.json")
            for cv_seed in cfg.cv_seeds:
                split_plan = make_split_plan(meta, cfg, cv_seed)
                split_id = digest(dict(data=data_identity, plan=split_plan))
                split_path = root / "splits" / f"sub-{subject:02d}_{split_id}.json"
                save_json(split_plan, split_path)
                for n_channels in cfg.channel_counts:
                    indices = subsets[n_channels]
                    subset_names = [names[i] for i in indices]
                    views = SubjectViews(X[:, indices], cfg.expected_sfreq)
                    for pipeline in cfg.pipelines:
                        for plan in split_plan["folds"]:
                            identity = dict(**scientific_identity(cfg, pipeline), subject=subject, channels=subset_names,
                                            split=split_id, fold=plan["fold"], data=data_identity,
                                            pretrained=pretrained[1] if pipeline == "labram" else None)
                            task_id = digest(identity)
                            task_dir = root / "checkpoints" / f"sub-{subject:02d}" / pipeline / f"{n_channels}ch" / task_id
                            task_dir.mkdir(parents=True, exist_ok=True)
                            completed = task_dir / "complete.json"
                            prediction_path = task_dir / "test_predictions.csv"
                            fold_seed = int(cv_seed + subject*100003 + plan["fold"]*1009)
                            if cfg.resume and checkpoint_complete(task_dir, task_id):
                                saved = json.loads(completed.read_text())
                                frame = pd.read_csv(prediction_path)
                                if saved["identity"] != task_id or set(frame.trial_id) != set(meta.trial_id.iloc[plan["test"]]):
                                    raise RuntimeError("Checkpoint identity or trial coverage mismatch")
                                fold_row = saved["fold_result"]
                                print(f"Resumed {pipeline} subject {subject:02d} {n_channels}ch fold {plan['fold']}")
                            else:
                                started = time.perf_counter()
                                save_json(identity, task_dir / "identity.json")
                                try:
                                    selected = select_candidate(candidate_list(cfg, pipeline), pipeline, views,
                                        meta.y_true.to_numpy(), subset_names, plan, cfg, fold_seed, task_dir,
                                        pretrained, data_identity)
                                    probability, train_metrics = refit_and_predict(selected, pipeline, views,
                                        meta.y_true.to_numpy(), subset_names, plan, cfg, fold_seed, task_dir,
                                        pretrained, data_identity)
                                except Exception as exc:
                                    save_json(dict(error=repr(exc), stage="selection_or_refit", identity=task_id), task_dir / "error.json")
                                    raise
                                test = np.array(plan["test"])
                                if probability.shape != (len(test), 4) or not np.isfinite(probability).all():
                                    raise ValueError("Invalid probability matrix")
                                frame = meta.iloc[test].copy()
                                frame["trial_index"] = test
                                frame["pipeline"], frame["n_channels"], frame["cv_seed"] = pipeline, n_channels, cv_seed
                                frame["outer_fold"] = plan["fold"]
                                frame["diagnostic"] = "real_labels" if cfg.label_permutation_seed is None else f"permuted_{cfg.label_permutation_seed}"
                                frame["y_pred"] = probability.argmax(1)
                                frame["candidate"] = digest(selected["spec"])
                                frame["checkpoint"] = relative_path(task_dir / "bundle.pkl", directory)
                                for c in CLASSES:
                                    frame[f"p_{c}"] = probability[:, c]
                                result = metrics(frame.y_true, frame.y_pred, probability)
                                fold_row = dict(pipeline=pipeline, subject=subject, n_channels=n_channels, cv_seed=cv_seed,
                                                outer_fold=plan["fold"], n_train=len(plan["train"]), n_test=len(test),
                                                diagnostic=frame.diagnostic.iloc[0], **result,
                                                **{"train_"+k: v for k, v in train_metrics.items()},
                                                selected_candidate=json.dumps(selected["spec"], sort_keys=True),
                                                svm_backend=selected["spec"].get("svm_backend", "not_svm"),
                                                refit_epochs=selected["refit_epochs"], seconds=time.perf_counter()-started,
                                                split_manifest=relative_path(split_path, directory))
                                save_csv(frame, prediction_path)
                                save_json(dict(identity=task_id, fold_result=fold_row, files=bundle_inventory(task_dir)), completed)
                                print(f"{pipeline} | subject {subject:02d} | {n_channels}ch | fold {plan['fold']}/{cfg.outer_folds} "
                                      f"| inner={selected['score']['balanced_accuracy']:.3f} | test={result['accuracy']:.3f}")
                            frames.append(frame); fold_rows.append(fold_row)
                            save_json(dict(status="running", finished_outer_tasks=len(fold_rows)), directory / "progress.json")
                        group = pd.concat(frames[-cfg.outer_folds:])
                        if len(group) != len(meta) or set(group.trial_id) != set(meta.trial_id) or group.trial_id.duplicated().any():
                            raise AssertionError("Outer OOF coverage is not exact")
                        # Write partial-but-complete subject groups for recovery during long runs.
                        result = write_reports(pd.concat(frames, ignore_index=True), directory, fold_rows)
                    del views
            del X
    save_json(dict(status="complete", finished_outer_tasks=len(fold_rows)), directory / "progress.json")
    print(f"Saved results: {directory}")
    return result


def run_permutation_control(cfg, n_permutations=1):
    """Re-run the complete selection procedure on within-session shuffled labels.
    One permutation is a smoke diagnostic, not a significance test. Exchangeability
    within sessions is an assumption; block-correlated data require block-aware nulls.
    """
    if n_permutations < 1:
        raise ValueError("n_permutations must be positive")
    results = []
    for repeat in range(n_permutations):
        result = run_experiments(replace(cfg, label_permutation_seed=901+repeat))
        results.append(result["summary"].assign(permutation=repeat))
    return pd.concat(results, ignore_index=True)


def plot_results(result):
    import matplotlib.pyplot as plt
    directory = Path(result["directory"]) / "figures"
    directory.mkdir(exist_ok=True, parents=True)
    subjects = result["subject_metrics"]
    fig, ax = plt.subplots(figsize=(9, 4))
    for (pipeline, channels), group in subjects.groupby(["pipeline", "n_channels"]):
        ax.plot(group.subject, 100*group.accuracy, marker="o", label=f"{pipeline} {channels}ch")
    ax.axhline(25, color="gray", linestyle="--", label="Chance (25%)")
    ax.set(xlabel="Subject", ylabel="Out-of-fold accuracy (%)", ylim=(0, 100))
    ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
    fig.tight_layout(); fig.savefig(directory / "subject_accuracy.png", dpi=180)
    plt.show()
    for (pipeline, channels), group in result["oof"].groupby(["pipeline", "n_channels"]):
        # Display average per-subject recall matrices, not trial-weighted large-subject results.
        matrices = [confusion_matrix(g.y_true, g.y_pred, labels=CLASSES, normalize="true")
                    for _, g in group.groupby(["subject", "cv_seed"])]
        matrix = np.mean(matrices, axis=0)
        fig, ax = plt.subplots(figsize=(4.8, 4.3))
        im = ax.imshow(matrix, vmin=0, vmax=1, cmap="Blues")
        for i, j in itertools.product(range(4), repeat=2):
            ax.text(j, i, f"{100*matrix[i,j]:.1f}%", ha="center", va="center", color="white" if matrix[i,j] > .5 else "black")
        ax.set(xticks=range(4), yticks=range(4), xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
               xlabel="Predicted", ylabel="True", title=f"{pipeline} · {channels} channels")
        fig.colorbar(im, ax=ax, label="Mean subject recall")
        fig.tight_layout(); fig.savefig(directory / f"confusion_{pipeline}_{channels}ch.png", dpi=180)
        plt.show()

In [ ]:
SOURCE_SHA256 = "3f677870655c580724f0fff5c7430dfc0e491163117020563c13eea2ce95828c"
print("Implementation:", VERSION)
print("Package versions:", package_versions())

## Build the configuration and inspect the run

Profiles change budgets and candidate sets. You can override individual fields below before
running. For example, `cfg.outer_folds=5` changes outer evaluation; `cfg.inner_folds=3`
changes internal selection. They are not train/validation/test percentages.

For a first full comparison, keep 128 channels. The optional reduced sets are fixed, nested,
geometry-based sets: they contain no label information and are shared by all models.
Task-optimized electrode selection is not part of this notebook's first experiment.

In [ ]:
enabled = tuple(name for name, enabled in (
    ("classical", RUN_CLASSICAL), ("eegnet", RUN_EEGNET), ("labram", RUN_LABRAM)
) if enabled)
cfg = make_config(
    PROFILE, data_root=DATA_ROOT, output_root=OUTPUT_ROOT,
    subject_ids=SUBJECT_IDS, channel_counts=CHANNEL_COUNTS,
    pipelines=enabled, cv_seeds=CV_SEEDS, resume=RESUME, classical_feature_set=CLASSICAL_FEATURE_SET, svm_backend=SVM_BACKEND,
)

# Optional overrides. Re-run this cell and the runner after editing.
# cfg.outer_folds = 5
# cfg.inner_folds = 3
# cfg.max_epochs = 100
# cfg.min_epochs = 20
# cfg.patience = 15
# cfg.ensemble_seeds = (0, 1, 2)
# cfg.labram_batch_size = 4  # reduce if a T4 runs out of memory
# cfg.labram_offline = True # only after restoring the pretrained cache

cfg.validate()
if Path(DATA_ROOT).exists():
    display(preview_run(cfg))
else:
    print("Set DATA_ROOT to the attached processed dataset before enabling the runner.")

for pipeline in cfg.pipelines:
    print(f"\n{pipeline} candidates:")
    display(pd.DataFrame(candidate_list(cfg, pipeline)))

## Optional CUDA check before the experiment

The runner automatically checks CUDA. You can also run this cell separately first.
The success message confirms that GPU binary fitting and four-class prediction were tested.


In [ ]:
RUN_CUDA_PREFLIGHT = False
if RUN_CUDA_PREFLIGHT and SVM_BACKEND == "cuda":
    CUDA_SVM_REPORT = cuda_svm_preflight()


## Optional feature-table preview

Enable this cell to inspect four original trials and the schema before training. This extracts
fixed features only; it does not fit a scaler, select features, or train a classifier. Each row
is one original trial. The full raw table has thousands of columns; only the first 12 are shown.


In [ ]:
SHOW_FEATURE_PREVIEW = False
if SHOW_FEATURE_PREVIEW:
    subjects = sorted({record[0] for record in discover_recordings(cfg.data_root)})
    subject = subjects[0] if cfg.subject_ids is None else cfg.subject_ids[0]
    preview_eeg, preview_names, preview_meta, _ = load_subject(cfg, subject)
    indices = channel_subsets(preview_names)[cfg.channel_counts[0]]
    names = [preview_names[index] for index in indices]
    feature_spec = next((spec for spec in candidate_list(cfg, "classical")
                         if spec["representation"] in ("eight_window", "bandpower_eight_window")), None)
    if feature_spec is None:
        print("Choose combined or eight_window to preview the new features.")
    else:
        preview_views = SubjectViews(preview_eeg[:4, indices], cfg.expected_sfreq)
        extractor = classical_extractor(feature_spec)
        matrix = extractor.fit_transform(classical_data(preview_views, feature_spec))
        schema = extractor.describe(names)
        table = pd.DataFrame(matrix[:, :12], columns=schema.feature_name.iloc[:12])
        table.insert(0, "trial_id", preview_meta.trial_id.iloc[:len(matrix)].to_numpy())
        print("Raw feature shape (original trials, columns):", matrix.shape)
        display(table)
        display(schema.head(24))
        display(pd.Series(extractor.metadata()))
        del preview_views, matrix, table, extractor
    del preview_eeg

## Easy runner — all enabled pipelines

The first LaBraM run downloads its public checkpoint. Pipelines run sequentially and share
saved split manifests. Running this cell again with identical settings resumes completed
work. All fitting is local to your Kaggle runtime/computer.

In [ ]:
RESULTS = None
if RUN_EXPERIMENTS:
    RESULTS = run_experiments(cfg)
    display(RESULTS["summary"])
else:
    print("Ready. Set RUN_EXPERIMENTS=True in Easy settings, then rerun the settings/configuration/runner cells.")

## Results and confusion matrices

Plots show exact trial-level OOF performance. The confusion plot averages subject recall
matrices so subjects with more trials do not dominate. In smoke mode these plots validate
the reporting path only. Confidence intervals require more than one subject.

In [ ]:
if RESULTS is not None:
    display(RESULTS["subject_metrics"])
    plot_results(RESULTS)

## Restore saved results without training

Set `SAVED_RUN` to one of the printed `.../runs/<id>` directories. These CSVs are sufficient
for tables and confusion matrices. Keep the complete output root to restore the models too.

In [ ]:
SAVED_RUN = None  # e.g. "/kaggle/working/inner_speech_eight_window_cuda/runs/<id>"
if SAVED_RUN is not None:
    saved = Path(SAVED_RUN)
    restored_results = {
        "directory": saved,
        "oof": pd.read_csv(saved / "oof_predictions.csv"),
        "subject_metrics": pd.read_csv(saved / "subject_metrics.csv"),
        "summary": pd.read_csv(saved / "summary.csv"),
    }
    display(restored_results["summary"])
    plot_results(restored_results)

## Restore a trained fold model without fitting

`predict_saved_fold` checks input sampling frequency and electrode order. It expects only
the `[1.0,3.5)` action interval in **microvolts**, not a full epoch and not volts. The bundle
knows the selected window, filters, normalization, learned transforms, architecture, and
ensemble. Restoring a frozen LaBraM probe uses its saved encoder and does not need another
Hub download. For evaluating new data, use recordings with the same preprocessing and schema.

```python
bundle_path = Path(".../checkpoints/sub-01/classical/128ch/<id>/bundle.pkl")
bundle = read_pickle(bundle_path)
probabilities = predict_saved_fold(
    bundle_path,
    action_eeg,                         # [trials, channels, 640] at 256 Hz; microvolts
    channel_names=bundle["channel_names"],
    sfreq=256.,
    cfg=cfg,
)
predicted_labels = probabilities.argmax(axis=1)
```

These are evaluation-fold models. Their stored test metrics do not become valid metrics
for a later model retrained using the test trials.

## Optional shuffled-label control

This reruns the **complete nested procedure** with labels shuffled within subject/session.
It is off by default because it can cost as much as another experiment. One shuffle is a
basic diagnostic, not a p-value or proof of no leakage. Formal permutation testing needs
many repeats and an exchangeability assumption appropriate to the recording's blocks.
Keep these outputs labeled as diagnostic; they are stored as separate runs.

In [ ]:
RUN_SHUFFLED_LABEL_CONTROL = False
if RUN_SHUFFLED_LABEL_CONTROL:
    null_results = run_permutation_control(cfg, n_permutations=1)
    display(null_results)

## Output map

```text
inner_speech_eight_window_cuda/
  pretrained/          pinned checkpoint, content hash, reusable Hub cache
  frozen_features/     deterministic embeddings and saved encoder for frozen probes
  splits/              original trial IDs, outer folds, and inner folds
  checkpoints/
    sub-XX/<pipeline>/<channels>/<identity>/
      identity.json
      inner_search/<candidate>/inner_oof.csv, inner_XX_history.csv, complete.json
      inner_leaderboard.csv
      selected.json
      bundle.pkl
      member_0.pt, member_0_refit_history.csv ...   # neural refits
      feature_extraction.json                      # eight-window parameters and counts
      raw_feature_schema.csv, selected_feature_schema.csv
      feature_reduction.npz                        # classical transforms also in bundle
      test_predictions.csv
      complete.json                               # checksums of required model files
  runs/<run-id>/
    run_config.json, progress.json
    data_audits/, channel_subsets.json
    oof_predictions.csv, fold_metrics.csv
    subject_seed_metrics.csv, subject_metrics.csv, summary.csv
    confusion_matrices/, classification_reports/, figures/
```

Changed scientific settings create a new run/cache identity. Downloading only `runs/`
preserves results but omits the reusable models and pretrained cache. A full run may create
many neural checkpoints. Monitor Kaggle disk space and preserve completed output before
the runtime is removed. Resume covers completed fits; it cannot restore an interrupted
epoch from its optimizer state.

## Method references and limitations

- [Thinking Out Loud dataset and trial timing](https://www.nature.com/articles/s41597-022-01147-2).
- [MNE multiclass CSP](https://mne.tools/stable/generated/mne.decoding.CSP.html).
- [pyRiemann tangent-space mapping](https://pyriemann.readthedocs.io/en/latest/generated/pyriemann.tangentspace.TangentSpace.html).
- [EEGNet authors' implementation](https://github.com/vlawhern/arl-eegmodels).
- [FBCNet authors' implementation](https://github.com/ravikiran-mane/FBCNet).
- [LaBraM authors' implementation](https://github.com/935963004/LaBraM) and
  [Braindecode checkpoint](https://huggingface.co/braindecode/labram-pretrained).

This is a new experimental implementation, not a reproduction of published scores.
Synthetic verification demonstrates execution, learning on a known artificial signal,
split integrity, and restoration. It cannot establish real inner-speech performance.
The exact CUDA/T4 environment must still be validated by your Kaggle smoke run.

CUDA SVM folds also save svm_backend.json with hardware and calibration details.